# `_chunk_scan_fwd` in JAX Pallas

## What this kernel computes

Given:
- `CB  (batch, nchunks, ngroups, chunk_size, chunk_size)` — pre-computed Gram matrix C@B^T
- `x   (batch, seqlen, nheads, hdim)` — SSM input
- `dt  (batch, nheads, nchunks, chunk_size)` — discretization step
- `dA  (batch, nheads, nchunks, chunk_size)` — cumulative log-decay (dA_cumsum)
- `C   (batch, seqlen, ngroups, dstate)` — output-projection vectors
- `states (batch, nchunks, nheads, hdim, dstate)` — per-chunk states (from `_state_passing_fwd`)

It produces `Y (batch, seqlen, nheads, hdim)` as the sum of two terms:

```
Y[b, c*Q+m, h, :] =

  Y_off:  exp(dA[b,h,c,m]) * C[b,c*Q+m, g, :] @ states[b,c, h, :, :]^T
          (inter-chunk: contribution of all previous chunks via propagated state)

+ Y_diag: sum_{k<=m}  CB[b,c,g,m,k]                          -- C@B^T at (m,k)
                     * exp(min(dA[b,h,c,m] - dA[b,h,c,k], 0)) -- intra-chunk decay m→k
                     * dt[b,h,c,k]                              -- ZOH discretization
                     * x[b, c*Q+k, h, :]                        -- input at position k
          (intra-chunk: lower-triangular scan, causal)
```

where `g = h // ratio` (GQA: `ratio = nheads // ngroups` heads share one B/C group).

## Connection to `ssd_minimal.py`

Yes — this kernel computes **exactly** `Y_diag + Y_off` from `ssd_minimal_discrete`:

| minimal.py | this kernel | note |
|---|---|---|
| `Y_off = einsum('bclhn,bchpn,bhcl->bclhp', C, states, exp(dA_cs))` | same | inter-chunk C@state |
| `Y_diag = einsum("bclhn,bcshn,bhcls,bcshp->bclhp", C, B, L, X)` | same | intra-chunk with L=CB*decay |

The difference is that this kernel takes **pre-computed** `CB = C@B^T` (from `_bmm_chunk_fwd`)
rather than computing the inner product inline.

## Pipeline position
```
_chunk_cumsum_fwd  →  dA_cumsum
_chunk_state_fwd   →  raw states
_state_passing_fwd →  propagated states    ← states input here
_bmm_chunk_fwd     →  CB                   ← CB input here
_chunk_scan_fwd    →  Y                    ← THIS KERNEL
```

In [1]:
import os, sys, types, importlib.util, time
import numpy as np

os.environ.setdefault(
    "CC",
    os.path.expanduser("~/miniconda3/envs/cutedsl/bin/x86_64-conda-linux-gnu-gcc"),
)

import jax
import jax.numpy as jnp
from jax import lax
import jax.experimental.pallas as pl
from jax._src.pallas.triton.core import CompilerParams

# ── load mamba modules via importlib (avoids __init__.py CUDA deps) ──────────
MAMBA_ROOT = os.path.expanduser("~/mamba")

def _stub_pkg(name, path=None):
    """Create a minimal package stub so sub-modules can set themselves up."""
    m = types.ModuleType(name)
    if path:
        m.__path__ = [path]
    m.__package__ = name
    sys.modules[name] = m
    return m

def _load_mod(name, relpath):
    """Load a real module from MAMBA_ROOT/relpath, registering it in sys.modules."""
    spec = importlib.util.spec_from_file_location(
        name, os.path.join(MAMBA_ROOT, relpath)
    )
    m = importlib.util.module_from_spec(spec)
    sys.modules[name] = m
    spec.loader.exec_module(m)
    return m

# Stub package hierarchy so relative imports resolve
_stub_pkg("mamba_ssm",            os.path.join(MAMBA_ROOT, "mamba_ssm"))
_stub_pkg("mamba_ssm.utils",      os.path.join(MAMBA_ROOT, "mamba_ssm/utils"))
_stub_pkg("mamba_ssm.ops",        os.path.join(MAMBA_ROOT, "mamba_ssm/ops"))
_stub_pkg("mamba_ssm.ops.triton", os.path.join(MAMBA_ROOT, "mamba_ssm/ops/triton"))

# Load real modules in dependency order
_load_mod("mamba_ssm.utils.determinism",    "mamba_ssm/utils/determinism.py")
_load_mod("mamba_ssm.ops.triton.ssd_bmm",   "mamba_ssm/ops/triton/ssd_bmm.py")
scan_mod = _load_mod("mamba_ssm.ops.triton.ssd_chunk_scan",
                     "mamba_ssm/ops/triton/ssd_chunk_scan.py")

import torch
from mamba_ssm.ops.triton.ssd_chunk_scan import _chunk_scan_fwd
from mamba_ssm.ops.triton.ssd_bmm import _bmm_chunk_fwd

def to_torch_bf16(x):
    """JAX bfloat16 → CUDA torch bfloat16 (via float32 to avoid ml_dtypes issue)."""
    return torch.from_numpy(np.array(x.astype(jnp.float32))).bfloat16().cuda()

def to_torch_f32(x):
    """JAX float32 → CUDA torch float32."""
    return torch.from_numpy(np.array(x.astype(jnp.float32))).cuda()

print(f"JAX     : {jax.__version__}")
print(f"devices : {jax.devices()}")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")

JAX     : 0.9.0.1


devices : [CudaDevice(id=0)]
PyTorch : 2.8.0+cu129
GPU     : NVIDIA GeForce RTX 4090


## Triton Reference: `_chunk_scan_fwd_kernel`

### Grid
```
grid = (
    cdiv(chunk_size, BLOCK_M) * cdiv(hdim, BLOCK_N),   # axis-0: output tiles (m, n)
    batch * nchunks,                                    # axis-1: (batch, chunk)
    nheads,                                             # axis-2: head
)
```
Each kernel instance writes one `(BLOCK_M, BLOCK_N)` tile of `Y[b, c*Q+m*BM : c*Q+(m+1)*BM, h, n*BN:(n+1)*BN]`.

### GQA pointer arithmetic
CB and C are indexed with `pid_h // ratio` (group index) rather than `pid_h` (head index):
```triton
cb_ptr += ... + (pid_h // nheads_ngroups_ratio) * stride_cb_head
C_ptr  += ... + (pid_h // nheads_ngroups_ratio) * stride_C_head
```

### `dstate` loop (`BLOCK_SIZE_DSTATE <= 128`)
When dstate fits in one tile, Y_off is computed in a single `tl.dot(C, prev_states)`.
For dstate > 128, it tiles over dstate with an inner loop.

### Inner K-loop (Y_diag) with `IS_CAUSAL=True`
```triton
K_MAX = min((pid_m + 1) * BLOCK_M, chunk_size)   # causal: only iterate up to row m
for k in range(0, K_MAX, BLOCK_K):
    cb  = load CB[m, k..k+BK]                     # (BM, BK) float32
    dA_k = load dA_cs[k..k+BK]                    # (BK,)
    # intra-chunk decay from position k to m:
    cb *= exp(min(dA_cs_m[:, None] - dA_k[None, :], 0))  # (BM, BK)
    cb *= dt_k                                            # (BK,)
    if IS_CAUSAL: apply lower-triangular mask             # zero out k > m
    x  = load x[k..k+BK, n..n+BN]                # (BK, BN) bfloat16
    acc += tl.dot(cb, x)                          # (BM, BN), fp32 accumulation
```
The `K_MAX` early exit avoids loading x for k tiles completely above the diagonal.

In [2]:
# Triton kernel (condensed annotated reference)
# Source: mamba/mamba_ssm/ops/triton/ssd_chunk_scan.py  lines 49–180
#
# @triton.autotune(configs=[BM={32..128}, BN={32..256}, BK={32..64}], key=['chunk_size','hdim','dstate'])
# @triton.jit
# def _chunk_scan_fwd_kernel(
#   cb_ptr,         # (batch, nchunks, ngroups, chunk_size, chunk_size)  f32
#   x_ptr,          # (batch, seqlen, nheads, hdim)                      bf16
#   z_ptr,          # (batch, seqlen, nheads, hdim)  or None             bf16
#   out_ptr,        # (batch, seqlen, nheads, hdim)                      bf16  [output]
#   out_x_ptr,      # (batch, seqlen, nheads, hdim)  or None             bf16  [pre-z output]
#   dt_ptr,         # (batch, nheads, nchunks, chunk_size)               f32
#   dA_cumsum_ptr,  # (batch, nheads, nchunks, chunk_size)               f32
#   seq_idx_ptr,    # (batch, seqlen)                or None             i32
#   C_ptr,          # (batch, seqlen, ngroups, dstate)                   bf16
#   prev_states_ptr,# (batch, nchunks, nheads, hdim, dstate)             f32
#   D_ptr,          # (nheads,) or (nheads, hdim)    or None             f32
# ):
#   # ── Decode grid indices ─────────────────────────────────────────────────
#   pid_bc = tl.program_id(axis=1)                      # encodes (batch, chunk)
#   pid_c  = pid_bc // batch
#   pid_b  = pid_bc - pid_c * batch
#   pid_h  = tl.program_id(axis=2)                      # head index
#   num_pid_n = tl.cdiv(hdim, BLOCK_SIZE_N)
#   pid_m  = tl.program_id(axis=0) // num_pid_n         # row tile in chunk_size
#   pid_n  = tl.program_id(axis=0) % num_pid_n          # col tile in hdim
#
#   # ── Advance pointers to this (batch, chunk, head) ──────────────────────
#   # GQA: CB and C use group index (pid_h // ratio) instead of pid_h
#   cb_ptr += pid_b*... + pid_c*chunk_size*... + (pid_h // ratio)*stride_cb_head
#   C_ptr  += pid_b*... + pid_c*chunk_size*... + (pid_h // ratio)*stride_C_head
#   x_ptr  += pid_b*... + pid_c*chunk_size*stride_x_seqlen + pid_h*stride_x_head
#   prev_states_ptr += pid_b*... + pid_c*stride_states_chunk + pid_h*stride_states_head
#
#   offs_m = pid_m * BLOCK_M + arange(0, BLOCK_M)       # row positions in chunk
#   offs_n = pid_n * BLOCK_N + arange(0, BLOCK_N)       # col positions in hdim
#   dA_cs_m = load(dA_cumsum + offs_m)                  # (BM,) cumulative decay at rows
#   acc = zeros((BLOCK_M, BLOCK_N), f32)
#
#   # ══ Y_off: inter-chunk contribution ════════════════════════════════════
#   # C[m, :] @ states[:, n] * exp(dA_cs_m)
#   # Shapes: (BM, dstate) @ (dstate, BN) → (BM, BN)
#   scale_m = exp(dA_cs_m)                              # (BM,)  or 0 for cross-doc
#   C       = load C_ptrs  # (BM, dstate) bf16
#   states  = load prev_states_ptrs  # (dstate, BN) f32
#   acc    += tl.dot(C, states.to(bf16)) * scale_m[:, None]
#   # (for dstate > 128: loop over dstate in BLOCK_SIZE_K chunks)
#
#   # ══ Y_diag: intra-chunk causal scan ════════════════════════════════════
#   K_MAX = min((pid_m + 1) * BLOCK_M, chunk_size)      # causal early exit
#   for k in range(0, K_MAX, BLOCK_K):
#       cb   = load CB[offs_m, k + offs_k]              # (BM, BK) f32
#       dA_k = load dA_cumsum[k + offs_k]               # (BK,)
#       dt_k = load dt[k + offs_k]                      # (BK,)
#       # intra-chunk decay: exp(min(dA_m[:, None] - dA_k[None, :], 0))
#       cb  *= exp(min(dA_cs_m[:, None] - dA_k[None, :], 0))  # (BM, BK)
#       cb  *= dt_k[None, :]                            # scale by ZOH step
#       if IS_CAUSAL: cb = where(offs_m[:,None] >= k+offs_k[None,:], cb, 0)
#       x    = load x[k + offs_k, offs_n]              # (BK, BN) bf16
#       acc += tl.dot(cb.to(bf16), x)                  # (BM, BN) fp32 accumulation
#
#   # ══ Optional: D residual, z gating ════════════════════════════════════
#   # if HAS_D: acc += x * D
#   # if HAS_Z: acc *= z * sigmoid(z)  [SiLU gating]
#
#   store(out_ptr + offs_m[:, None] * stride_seqlen + offs_n[None, :], acc)

print("Triton kernel annotated above")

Triton kernel annotated above


## Naive JAX

Uses `jnp.matmul` for the two GEMM terms. No custom kernel, no seq_idx, no D/z.

For `Y_diag` we use pre-computed `CB` (same as what Triton takes as input) rather than
re-deriving `C @ B^T` — this keeps the comparison apples-to-apples for the scan step.
The intra-chunk decay matrix `L[b,h,c,m,k] = exp(min(dA[m]-dA[k], 0)) * dt[k] * causal_mask[m,k]`
is computed explicitly and applied to `CB` before the matmul.

In [3]:
def chunk_scan_naive(CB, x, dt, dA_cumsum, C, states):
    """
    Minimal JAX implementation of _chunk_scan_fwd.

    CB:         (batch, nchunks, ngroups, chunk_size, chunk_size)  float32
    x:          (batch, seqlen, nheads, hdim)                      bfloat16
    dt:         (batch, nheads, nchunks, chunk_size)               float32
    dA_cumsum:  (batch, nheads, nchunks, chunk_size)               float32
    C:          (batch, seqlen, ngroups, dstate)                   bfloat16
    states:     (batch, nchunks, nheads, hdim, dstate)             float32
    Returns:    (batch, seqlen, nheads, hdim)                      bfloat16
    """
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    # ── Reshape into (batch, nchunks, nheads, chunk_size, ...) ──────────────
    # x_c: (B, C, H, Q, P)
    x_c = x.reshape(batch, nchunks, chunk_size, nheads, hdim).transpose(0, 1, 3, 2, 4)

    # C_c: (B, C, H, Q, N) — expand ngroups → nheads
    C_c = C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
    if ratio > 1:
        C_c = jnp.repeat(C_c, ratio, axis=3)          # (B, C, Q, H, N)
    C_c = C_c.transpose(0, 1, 3, 2, 4)                 # (B, C, H, Q, N)

    # ── Y_off: inter-chunk ───────────────────────────────────────────────────
    # state_decay = exp(dA_cumsum): (B, H, C, Q) → (B, C, H, Q)
    state_decay = jnp.exp(dA_cumsum).transpose(0, 2, 1, 3)   # (B, C, H, Q)
    # C_scaled[b,c,h,m,n] = C[b,c,h,m,n] * exp(dA[b,h,c,m])
    C_scaled = C_c.astype(jnp.float32) * state_decay[..., None]  # (B, C, H, Q, N)
    # Y_off = C_scaled @ states^T:  (B,C,H,Q,N) @ (B,C,H,N,P) → (B,C,H,Q,P)
    Y_off = jnp.matmul(C_scaled, states.transpose(0, 1, 2, 4, 3).astype(jnp.float32))

    # ── Y_diag: intra-chunk causal scan ─────────────────────────────────────
    # Intra-chunk decay: L[b,h,c,m,k] = exp(min(dA[m]-dA[k], 0))  (B,H,C,Q,Q)
    dA_m = dA_cumsum[..., :, None]   # (B, H, C, Q, 1)
    dA_k = dA_cumsum[..., None, :]   # (B, H, C, 1, Q)
    L = jnp.exp(jnp.minimum(dA_m - dA_k, 0.0))                   # (B, H, C, Q, Q)
    # Causal mask (lower triangular, m >= k)
    causal = jnp.tril(jnp.ones((chunk_size, chunk_size), dtype=bool))
    L = jnp.where(causal, L, 0.0)                                  # (B, H, C, Q, Q)
    # Scale by dt: dt[b,h,c,k] multiplies along the k (column) dimension
    L = L * dt[..., None, :]           # (B, H, C, Q, Q):  [..., m, k] * dt[..., k]

    # Expand CB from ngroups → nheads and apply L
    CB_h = jnp.repeat(CB, ratio, axis=2) if ratio > 1 else CB     # (B, C, H, Q, Q)
    L_t  = L.transpose(0, 2, 1, 3, 4)                             # (B, C, H, Q, Q)
    CB_scaled = CB_h.astype(jnp.float32) * L_t                    # (B, C, H, Q, Q)

    # Y_diag = CB_scaled @ x_c:  (B,C,H,Q,Q) @ (B,C,H,Q,P) → (B,C,H,Q,P)
    Y_diag = jnp.matmul(CB_scaled, x_c.astype(jnp.float32))

    # ── Combine and reshape ──────────────────────────────────────────────────
    # (B,C,H,Q,P) → transpose → (B,C,Q,H,P) → reshape → (B,L,H,P)
    Y = (Y_diag + Y_off).transpose(0, 1, 3, 2, 4).reshape(batch, seqlen, nheads, hdim)
    return Y.astype(jnp.bfloat16)

print("chunk_scan_naive defined")

chunk_scan_naive defined


## Pallas Kernel Design

### Grid: `(BCH, PM, PN)`
```
BCH = batch * nchunks * nheads     — one slot per (batch, chunk, head)
PM  = chunk_size // BLOCK_M        — row tiles of the output Q-dimension
PN  = hdim // BLOCK_N              — col tiles of the output P-dimension
```
Each instance writes one `(BLOCK_M, BLOCK_N)` tile.

### Pre-flatten reshapes (wrapper)

| Tensor | Original shape | Flat shape | Reshape |
|--------|----------------|------------|---------|
| `x`    | `(B,L,H,P)` | `(BCH,Q,P)` | reshape+transpose |
| `CB`   | `(B,C,G,Q,Q)` | `(BCG,Q,Q)` | simple reshape |
| `C`    | `(B,L,G,N)` | `(BCG,Q,N)` | reshape+transpose |
| `dt`   | `(B,H,C,Q)` | `(BCH,Q)` | transpose+reshape |
| `dA`   | `(B,H,C,Q)` | `(BCH,Q)` | transpose+reshape |
| `states` | `(B,C,H,P,N)` | `(BCH,P,N)` | simple reshape |
| `out`  | `(B,L,H,P)` | `(BCH,Q,P)` | same as x |

where `BCG = batch * nchunks * ngroups`, `Q = chunk_size`.

### BlockSpecs

| Ref | Block shape | Index map | Pattern |
|-----|------------|-----------|--------|
| `cb`     | `(1, BM, Q)` | `(bcg, pm, 0)` | load row tile + all K cols |
| `C`      | `(1, BM, N)` | `(bcg, pm, 0)` | row tile, full dstate |
| `x`      | `(1, Q, BN)` | `(bch, 0, pn)` | full chunk rows, col tile |
| `dt`     | `(1, Q)`     | `(bch, 0)`     | full chunk dt |
| `dA_m`   | `(1, BM)`    | `(bch, pm)`    | just the BM m-rows of dA |
| `dA`     | `(1, Q)`     | `(bch, 0)`     | full chunk dA (for decay) |
| `states` | `(1, BN, N)` | `(bch, pn, 0)` | col tile, full dstate |
| `out`    | `(1, BM, BN)`| `(bch, pm, pn)`| output tile |

`dA` is passed **twice**: once as `(1, BM)` for `scale_m = exp(dA_m)` (Y_off),
once as `(1, Q)` for the full decay matrix (Y_diag).  
GQA mapping: `bcg = (bch // nheads) * ngroups + (bch % nheads) // ratio`.

### Why no inner K-loop?

Loading the full `(BLOCK_M, chunk_size)` CB and `(chunk_size, BLOCK_N)` x at once
enables a single `jnp.matmul` for Y_diag — no sequential K-tiling needed.
For `chunk_size ≤ 256` and `BLOCK_M ≤ 64` the tiles fit comfortably in registers.

Pallas has no `num_stages` software pipelining primitive, so a K-loop would NOT help
with latency hiding — the single large matmul is strictly better.

The causal mask `m_abs[:, None] >= k_abs[None, :]` is applied to `CB` before the matmul.
`m_abs = pl.program_id(1) * BLOCK_M + arange(BLOCK_M)` — arithmetic on a traced int
is fine in JAX; only *indexing* with a traced int fails.

In [4]:
def _make_chunk_scan_kernel(BLOCK_M, BLOCK_N, chunk_size):
    """
    Factory that captures tile sizes and chunk_size as compile-time constants.

    Refs (in order):
      cb_ref     (1, BLOCK_M, chunk_size)  f32   — row tile of CB, full K cols
      C_ref      (1, BLOCK_M, dstate)      bf16  — row tile of C
      x_ref      (1, chunk_size, BLOCK_N)  bf16  — full chunk rows, col tile
      dt_ref     (1, chunk_size)           f32   — full chunk dt
      dA_m_ref   (1, BLOCK_M)              f32   — dA_cumsum at the m-rows
      dA_ref     (1, chunk_size)           f32   — dA_cumsum full chunk (for decay)
      states_ref (1, BLOCK_N, dstate)      f32   — col tile of states
      out_ref    (1, BLOCK_M, BLOCK_N)     f32   — output tile (cast to bf16 in wrapper)
    """
    def _chunk_scan_fwd_kernel(
        cb_ref, C_ref, x_ref, dt_ref, dA_m_ref, dA_ref, states_ref, out_ref
    ):
        # ── Load all tiles ──────────────────────────────────────────────────
        # ref[0, :, :] = ref[scalar, full, full] — confirmed-safe Pallas access
        cb     = cb_ref[0, :, :]        # (BM, Q)  f32 — CB row tile, all K cols
        C      = C_ref[0, :, :]         # (BM, N)  bf16
        x      = x_ref[0, :, :]         # (Q, BN)  bf16
        dt     = dt_ref[0, :]           # (Q,)     f32
        dA_m   = dA_m_ref[0, :]         # (BM,)    f32 — cumsum at row positions
        dA     = dA_ref[0, :]           # (Q,)     f32 — cumsum at all K positions
        states = states_ref[0, :, :]    # (BN, N)  f32

        # ══ Y_off: inter-chunk contribution ════════════════════════════════
        # acc[m, p] = exp(dA_m[m]) * sum_n C[m,n] * states[p,n]
        #           = scale_m[m] * (C @ states^T)[m, p]
        # Triton: acc = tl.dot(C, prev_states.to(bf16)) * scale_m[:, None]
        scale_m = jnp.exp(dA_m)                                    # (BM,)
        acc = (
            jnp.matmul(C.astype(jnp.float32), states.astype(jnp.float32).T)
            * scale_m[:, None]
        )  # (BM, N) @ (N, BN) → (BM, BN)  f32

        # ══ Y_diag: intra-chunk causal scan ════════════════════════════════
        # decay[m, k] = exp(min(dA_m[m] - dA[k], 0))  — intra-chunk decay m→k
        # Triton: cb *= exp(tl.minimum(dA_cs_m[:,None] - dA_k[None,:], 0))
        decay = jnp.exp(
            jnp.minimum(dA_m[:, None] - dA[None, :], 0.0)
        )  # (BM, Q)  f32

        # Scale CB by decay and dt (ZOH discretization):
        # Triton: cb *= dt_k  then cast to bf16 before tl.dot
        cb_scaled = cb * decay * dt[None, :]    # (BM, Q)  f32

        # Causal mask: only positions k <= m contribute (IS_CAUSAL=True always)
        # m_abs[i] = pm * BLOCK_M + i   — absolute row position in chunk
        # Triton: K_MAX = min((pid_m+1)*BM, chunk_size) early-exits the loop;
        #         within the last tile: where(offs_m[:,None] >= k+offs_k[None,:], cb, 0)
        # Pallas: compute causal mask from program_id — arithmetic on traced int is fine
        pm    = pl.program_id(axis=1)                              # traced int32
        m_abs = pm * BLOCK_M + jnp.arange(BLOCK_M)                # (BM,) traced
        k_abs = jnp.arange(chunk_size)                             # (Q,)  static
        causal = m_abs[:, None] >= k_abs[None, :]                  # (BM, Q) bool
        cb_scaled = jnp.where(causal, cb_scaled, 0.0)              # (BM, Q)  f32

        # Y_diag = cb_scaled @ x:  (BM, Q) @ (Q, BN) → (BM, BN)
        # Triton: acc += tl.dot(cb.to(bf16), x)  — fp32 accumulation
        # Pallas: bf16 matmul unsupported on Triton backend — use f32 @ f32
        acc = acc + jnp.matmul(cb_scaled, x.astype(jnp.float32))  # (BM, BN)  f32

        # ── Write output ────────────────────────────────────────────────────
        # Output dtype is float32; wrapper casts to bfloat16 after reshape
        out_ref[0, :, :] = acc    # (BM, BN)  f32

    return _chunk_scan_fwd_kernel

print("_make_chunk_scan_kernel defined")

_make_chunk_scan_kernel defined


In [5]:
def chunk_scan_pallas(CB, x, dt, dA_cumsum, C, states, BLOCK_M=32, BLOCK_N=64):
    """
    Pallas implementation of _chunk_scan_fwd.

    CB:         (batch, nchunks, ngroups, chunk_size, chunk_size)  float32
    x:          (batch, seqlen, nheads, hdim)                      bfloat16
    dt:         (batch, nheads, nchunks, chunk_size)               float32
    dA_cumsum:  (batch, nheads, nchunks, chunk_size)               float32
    C:          (batch, seqlen, ngroups, dstate)                   bfloat16
    states:     (batch, nchunks, nheads, hdim, dstate)             float32
    Returns:    (batch, seqlen, nheads, hdim)                      bfloat16
    """
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    BCH = batch * nchunks * nheads    # flat (batch, chunk, head) index space
    BCG = batch * nchunks * ngroups   # flat (batch, chunk, group) index space
    PM  = chunk_size // BLOCK_M
    PN  = hdim // BLOCK_N

    # ── Pre-flatten inputs ──────────────────────────────────────────────────
    #
    # Convention: bch = b*(nchunks*nheads) + c*nheads + h
    #             bcg = b*(nchunks*ngroups) + c*ngroups + g = b*(nchunks*ngroups) + c*ngroups + h//ratio
    #
    # x: (B, L, H, P) → (B, C, Q, H, P) → (B, C, H, Q, P) → (BCH, Q, P)
    x_flat = (
        x.reshape(batch, nchunks, chunk_size, nheads, hdim)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCH, chunk_size, hdim)
    )
    # CB: (B, C, G, Q, Q) → (BCG, Q, Q)   [already contiguous in leading dims]
    CB_flat = CB.reshape(BCG, chunk_size, chunk_size)

    # C: (B, L, G, N) → (B, C, Q, G, N) → (B, C, G, Q, N) → (BCG, Q, N)
    C_flat = (
        C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCG, chunk_size, dstate)
    )
    # dt:  (B, H, C, Q) → (B, C, H, Q) → (BCH, Q)
    dt_flat  = dt.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    # dA:  (B, H, C, Q) → (BCH, Q)  (same layout as dt)
    dA_flat  = dA_cumsum.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    # states: (B, C, H, P, N) → (BCH, P, N)   [C, H are already adjacent after cumsum]
    states_flat = states.reshape(BCH, hdim, dstate)

    # ── BlockSpecs ──────────────────────────────────────────────────────────
    _nh = nheads; _ng = ngroups; _r = ratio
    _Q  = chunk_size; _N  = dstate

    # GQA index: bcg = (bch // nheads) * ngroups + (bch % nheads) // ratio
    # Mirrors Triton: (pid_h // nheads_ngroups_ratio) for cb_ptr and C_ptr
    def bcg(bch, pm, pn):
        return (bch // _nh) * _ng + (bch % _nh) // _r

    in_specs = [
        # CB_flat: (BCG, Q, Q)  block (1, BM, Q) — row tile, full K cols
        # Triton: cb_ptr += ... + (pid_h//ratio)*stride_cb_head
        pl.BlockSpec((1, BLOCK_M, _Q),  lambda bch, pm, pn: (bcg(bch, pm, pn), pm, 0)),

        # C_flat: (BCG, Q, N)  block (1, BM, N) — row tile, full dstate
        # Triton: C_ptr += ... + (pid_h//ratio)*stride_C_head
        pl.BlockSpec((1, BLOCK_M, _N),  lambda bch, pm, pn: (bcg(bch, pm, pn), pm, 0)),

        # x_flat: (BCH, Q, P)  block (1, Q, BN) — full chunk rows, col tile
        # Triton: x_ptr += ... + pid_h*stride_x_head; loads (BK, BN) in inner loop
        pl.BlockSpec((1, _Q, BLOCK_N),  lambda bch, pm, pn: (bch, 0, pn)),

        # dt_flat: (BCH, Q)  block (1, Q) — full chunk dt
        pl.BlockSpec((1, _Q),           lambda bch, pm, pn: (bch, 0)),

        # dA_m: (BCH, Q) viewed as (BCH, Q), block (1, BM) at (bch, pm)
        # Gives exactly dA_cumsum[bch, pm*BM : (pm+1)*BM] for scale_m
        # Triton: dA_cs_m = load(dA_cumsum + offs_m * stride)  (BM,)
        pl.BlockSpec((1, BLOCK_M),      lambda bch, pm, pn: (bch, pm)),

        # dA_full: same tensor, full Q — needed for decay[m,k] = exp(min(dA_m-dA_k,0))
        # Triton: dA_cs_k = load(dA_cumsum + offs_k)  loaded inside the K-loop
        pl.BlockSpec((1, _Q),           lambda bch, pm, pn: (bch, 0)),

        # states_flat: (BCH, P, N)  block (1, BN, N) — col tile, full dstate
        # Triton: prev_states_ptr += ... + pid_h*stride_states_head
        pl.BlockSpec((1, BLOCK_N, _N),  lambda bch, pm, pn: (bch, pn, 0)),
    ]
    out_specs = [
        # out: (BCH, Q, P)  block (1, BM, BN) — output tile (float32)
        pl.BlockSpec((1, BLOCK_M, BLOCK_N), lambda bch, pm, pn: (bch, pm, pn)),
    ]

    # dA_flat is passed TWICE: once sliced to BM (for scale_m), once full Q (for decay)
    out_flat, = pl.pallas_call(
        _make_chunk_scan_kernel(BLOCK_M, BLOCK_N, chunk_size),
        out_shape=[jax.ShapeDtypeStruct((BCH, chunk_size, hdim), jnp.float32)],
        in_specs=in_specs,
        out_specs=out_specs,
        grid=(BCH, PM, PN),
        compiler_params=CompilerParams(),
    )(CB_flat, C_flat, x_flat, dt_flat, dA_flat, dA_flat, states_flat)

    # ── Reshape output: (BCH, Q, P) → (B, C, H, Q, P) → (B, C, Q, H, P) → (B, L, H, P)
    out = (
        out_flat
        .reshape(batch, nchunks, nheads, chunk_size, hdim)
        .transpose(0, 1, 3, 2, 4)
        .reshape(batch, seqlen, nheads, hdim)
    )
    return out.astype(jnp.bfloat16)

print("chunk_scan_pallas defined")

chunk_scan_pallas defined


## Correctness Check

In [6]:
def make_inputs(batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size, seed=0):
    """Generate random JAX inputs matching _chunk_scan_fwd's expected shapes."""
    nchunks = seqlen // chunk_size
    ks = jax.random.split(jax.random.PRNGKey(seed), 6)
    x      = jax.random.normal(ks[0], (batch, seqlen,  nheads,  hdim),   dtype=jnp.bfloat16)
    C      = jax.random.normal(ks[1], (batch, seqlen,  ngroups, dstate), dtype=jnp.bfloat16)
    B      = jax.random.normal(ks[2], (batch, seqlen,  ngroups, dstate), dtype=jnp.bfloat16)
    dt     = jax.nn.softplus(jax.random.normal(ks[3], (batch, nheads, nchunks, chunk_size)))
    dA_cs  = -jnp.cumsum(
        jax.random.uniform(ks[4], (batch, nheads, nchunks, chunk_size)) * dt, axis=-1
    )  # negative cumulative sum → monotone decreasing (valid log-decay)
    states = jax.random.normal(ks[5], (batch, nchunks, nheads, hdim, dstate))
    # Compute CB with naive matmul (same as the benchmark's CB)
    B_c    = B.reshape(batch, nchunks, chunk_size, ngroups, dstate).transpose(0,1,3,2,4)
    C_c    = C.reshape(batch, nchunks, chunk_size, ngroups, dstate).transpose(0,1,3,2,4)
    CB     = jnp.matmul(
        C_c.astype(jnp.float32),
        B_c.transpose(0,1,2,4,3).astype(jnp.float32)
    )  # (batch, nchunks, ngroups, chunk_size, chunk_size)
    return dict(CB=CB, x=x, dt=dt, dA=dA_cs, C=C, B=B, states=states)

print("make_inputs defined")

make_inputs defined


In [7]:
batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = 2, 512, 8, 64, 64, 1, 64
inp = make_inputs(batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size)
CB, x, dt, dA, C, B, states = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

# ── Triton reference ─────────────────────────────────────────────────────────
CB_t  = to_torch_f32(CB);  x_t  = to_torch_bf16(x)
dt_t  = to_torch_f32(dt);  dA_t = to_torch_f32(dA)
C_t   = to_torch_bf16(C);  st_t = to_torch_f32(states)
Y_tri, _ = _chunk_scan_fwd(CB_t, x_t, dt_t, dA_t, C_t, st_t)
# Triton output is bfloat16; convert via float32 (BF16→FP32 is exact)
Y_ref = jnp.array(Y_tri.float().cpu().numpy()).astype(jnp.bfloat16)
print(f"Y shape: {Y_ref.shape}   |Y|_max: {float(jnp.abs(Y_ref.astype(jnp.float32)).max()):.1f}")

# ── Naive ────────────────────────────────────────────────────────────────────
Y_naive = jax.jit(lambda *a: chunk_scan_naive(*a))(CB, x, dt, dA, C, states)
diff_naive = float(jnp.max(jnp.abs(Y_naive.astype(jnp.float32) - Y_ref.astype(jnp.float32))))
print(f"Naive  vs Triton — max abs diff: {diff_naive:.3f}")

# ── Pallas ───────────────────────────────────────────────────────────────────
Y_pal = jax.jit(
    lambda *a: chunk_scan_pallas(*a, BLOCK_M=32, BLOCK_N=64)
)(CB, x, dt, dA, C, states)
diff_pal   = float(jnp.max(jnp.abs(Y_pal.astype(jnp.float32) - Y_ref.astype(jnp.float32))))
diff_np    = float(jnp.max(jnp.abs(Y_pal.astype(jnp.float32) - Y_naive.astype(jnp.float32))))
print(f"Pallas vs Triton — max abs diff: {diff_pal:.3f}")
print(f"Pallas vs Naive  — max abs diff: {diff_np:.4f}  (should be <0.5)")

# ── Notes on expected precision ──────────────────────────────────────────────
# Triton kernel casts cb_scaled to BF16 before tl.dot(cb.to(bf16), x).
# Our Pallas/naive use F32 @ F32 — strictly more accurate.
# At output magnitude 64-128, BF16 step = 0.5, so ≤1 BF16 ULP difference vs Triton is expected.
# Threshold: 2.0 = 2 BF16 steps at magnitude ≤256.
THRESHOLD = 2.0
assert diff_naive < THRESHOLD, f"Naive failed: {diff_naive}"
assert diff_pal   < THRESHOLD, f"Pallas failed: {diff_pal}"
assert diff_np    < 0.5,       f"Naive vs Pallas too large: {diff_np}"
print(f"\nAll assertions passed (threshold = {THRESHOLD})")

Y shape: (2, 512, 8, 64)   |Y|_max: 193.0


Naive  vs Triton — max abs diff: 0.500


Pallas vs Triton — max abs diff: 0.500
Pallas vs Naive  — max abs diff: 0.2500  (should be <0.5)

All assertions passed (threshold = 2.0)


In [8]:
# Test Nemotron config: ngroups=8, ratio=16
inp2 = make_inputs(1, 256, 128, 64, 128, 8, 256)
CB2, x2, dt2, dA2, C2, B2, st2 = [inp2[k] for k in ('CB','x','dt','dA','C','B','states')]

Y_ref2, _ = _chunk_scan_fwd(
    to_torch_f32(CB2), to_torch_bf16(x2), to_torch_f32(dt2),
    to_torch_f32(dA2), to_torch_bf16(C2), to_torch_f32(st2)
)
Y_ref2 = jnp.array(Y_ref2.float().cpu().numpy()).astype(jnp.bfloat16)

Y_pal2 = jax.jit(
    lambda *a: chunk_scan_pallas(*a, BLOCK_M=64, BLOCK_N=64)
)(CB2, x2, dt2, dA2, C2, st2)

Y_naive2 = jax.jit(lambda *a: chunk_scan_naive(*a))(CB2, x2, dt2, dA2, C2, st2)

diff_pal2   = float(jnp.max(jnp.abs(Y_pal2.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
diff_naive2 = float(jnp.max(jnp.abs(Y_naive2.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
diff_np2    = float(jnp.max(jnp.abs(Y_pal2.astype(jnp.float32) - Y_naive2.astype(jnp.float32))))
ymax2 = float(jnp.abs(Y_ref2.astype(jnp.float32)).max())

print(f"Nemotron (H=128, G=8, cs=256) |Y|_max: {ymax2:.1f}")
print(f"  Naive  vs Triton: {diff_naive2:.3f}")
print(f"  Pallas vs Triton: {diff_pal2:.3f}")
print(f"  Pallas vs Naive:  {diff_np2:.4f}")
assert diff_pal2 < 2.0 and diff_naive2 < 2.0 and diff_np2 < 0.5
print("  OK")

Nemotron (H=128, G=8, cs=256) |Y|_max: 276.0
  Naive  vs Triton: 1.000
  Pallas vs Triton: 1.000
  Pallas vs Naive:  0.2500
  OK


## Autotune: `(BLOCK_M, BLOCK_N)`

In [9]:
_cfg = dict(batch=1, seqlen=2048, nheads=32, hdim=64, dstate=64, ngroups=1, chunk_size=64)
_inp = make_inputs(**_cfg)
_CB, _x, _dt, _dA, _C, _st = [_inp[k] for k in ('CB','x','dt','dA','C','states')]

N_WARMUP, N_RUNS = 5, 20
BEST_BM, BEST_BN = 32, 64  # defaults; updated below
best_ms = float('inf')

print(f"Autotuning for: {_cfg}")
print(f"  {'BM':>4} {'BN':>4}     ms")
print("-" * 28)
for BM in [16, 32, 64]:
    for BN in [16, 32, 64]:
        if _cfg['chunk_size'] % BM != 0 or _cfg['hdim'] % BN != 0:
            continue
        fn = jax.jit(lambda cb,x,dt,da,c,st: chunk_scan_pallas(cb,x,dt,da,c,st,BLOCK_M=BM,BLOCK_N=BN))
        for _ in range(N_WARMUP): fn(_CB,_x,_dt,_dA,_C,_st).block_until_ready()
        t0 = time.perf_counter()
        for _ in range(N_RUNS): fn(_CB,_x,_dt,_dA,_C,_st).block_until_ready()
        ms = (time.perf_counter()-t0)/N_RUNS*1e3
        tag = "  <-- best" if ms < best_ms else ""
        print(f"  {BM:4d} {BN:4d}  {ms:6.3f} ms{tag}")
        if ms < best_ms:
            best_ms, BEST_BM, BEST_BN = ms, BM, BN

print(f"\nBest: BLOCK_M={BEST_BM}, BLOCK_N={BEST_BN}")

Autotuning for: {'batch': 1, 'seqlen': 2048, 'nheads': 32, 'hdim': 64, 'dstate': 64, 'ngroups': 1, 'chunk_size': 64}
    BM   BN     ms
----------------------------


    16   16   1.149 ms  <-- best
    16   32   1.157 ms


    16   64   1.149 ms  <-- best
    32   16   1.146 ms  <-- best


    32   32   1.135 ms  <-- best


    32   64   1.157 ms
    64   16   1.152 ms


    64   32   1.177 ms


    64   64   1.157 ms

Best: BLOCK_M=32, BLOCK_N=32


## Benchmarks

**Scan-only**: CB is pre-computed and not counted — isolates the scan kernel.

**Full pipeline**: includes CB computation (naive matmul) for Pallas;
for Triton, uses Triton's `_bmm_chunk_fwd` (which is faster than naive for some configs).

All timings use `fori_loop` amortization to eliminate JAX Python dispatch overhead (~1 ms).

In [10]:
SHAPE_SWEEP = [
    # (batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size),   label
    ((1, 2048, 32,  64,  64, 1,  64),  "Mamba2 standard   B=1 H=32  P=64  N=64  Q=64 "),
    ((1, 2048, 32,  64,  64, 1, 256),  "Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256"),
    ((1, 2048, 128, 64, 128, 8, 256),  "Nemotron          B=1 H=128 P=64  N=128 Q=256"),
    ((2, 2048, 32,  64,  64, 1,  64),  "Batched           B=2 H=32  P=64  N=64  Q=64 "),
]

N_AMORTIZE = 200

def _best_block(chunk_size, hdim):
    """Return best (BM, BN) from autotune result, fallback to largest valid."""
    if chunk_size % BEST_BM == 0 and hdim % BEST_BN == 0:
        return BEST_BM, BEST_BN
    bm = max(b for b in [16,32,64] if chunk_size % b == 0)
    bn = max(b for b in [16,32,64] if hdim % b == 0)
    return bm, bn

def bmm_naive_jax(C, B, chunk_size):
    """Compute CB with naive jnp.matmul (no custom kernel)."""
    batch, seqlen, ngroups, dstate = C.shape
    nchunks = seqlen // chunk_size
    C_c = C.reshape(batch,nchunks,chunk_size,ngroups,dstate).transpose(0,1,3,2,4)
    B_c = B.reshape(batch,nchunks,chunk_size,ngroups,dstate).transpose(0,1,3,2,4)
    return jnp.matmul(C_c.astype(jnp.float32), B_c.transpose(0,1,2,4,3).astype(jnp.float32))

def make_amortized(fn, n):
    @jax.jit
    def inner(*args):
        out0 = jnp.zeros_like(fn(*args))
        return lax.fori_loop(0, n, lambda i, _: fn(*args), out0)
    return inner

def bench_amortized(fn_am, args, n, warmup=3, runs=5):
    fn_am(*args).block_until_ready()
    for _ in range(warmup): fn_am(*args).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(runs): fn_am(*args).block_until_ready()
    return (time.perf_counter()-t0)/runs/n*1e3

def bench_triton_amortized(fn_t, args_t, n, warmup=10, runs=5):
    """Triton amortized: plain Python loop (Triton has low dispatch overhead)."""
    for _ in range(warmup): fn_t(*args_t)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n*runs): fn_t(*args_t)
    torch.cuda.synchronize()
    return (time.perf_counter()-t0)/runs/n*1e3

print("Benchmark utilities defined")

Benchmark utilities defined


In [11]:
print("═" * 82)
print("SCAN-ONLY benchmark  (CB pre-computed, not counted)")
print(f"{'Config':<45} {'Triton':>8} {'Naive':>8} {'Pallas':>8}")
print(f"{'':45} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8}")
print("─" * 82)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    CB, x, dt, dA, C, B, st = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

    # Triton: _chunk_scan_fwd (CB already float32 torch tensor)
    CB_t = to_torch_f32(CB); x_t = to_torch_bf16(x)
    dt_t = to_torch_f32(dt); dA_t = to_torch_f32(dA)
    C_t  = to_torch_bf16(C); st_t = to_torch_f32(st)
    t_tri = bench_triton_amortized(
        lambda: _chunk_scan_fwd(CB_t, x_t, dt_t, dA_t, C_t, st_t), (), N_AMORTIZE
    )

    # Naive JAX scan
    fn_naive_am = make_amortized(lambda cb,xj,dtj,daj,cj,sj: chunk_scan_naive(cb,xj,dtj,daj,cj,sj), N_AMORTIZE)
    t_naive = bench_amortized(fn_naive_am, (CB,x,dt,dA,C,st), N_AMORTIZE)

    # Pallas scan
    BM, BN = _best_block(chunk_size, hdim)
    fn_pal_am = make_amortized(lambda cb,xj,dtj,daj,cj,sj: chunk_scan_pallas(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_N=BN), N_AMORTIZE)
    t_pal = bench_amortized(fn_pal_am, (CB,x,dt,dA,C,st), N_AMORTIZE)

    print(f"{label:<45} {t_tri:>8.3f} {t_naive:>8.3f} {t_pal:>8.3f}")

print("═" * 82)

══════════════════════════════════════════════════════════════════════════════════
SCAN-ONLY benchmark  (CB pre-computed, not counted)
Config                                          Triton    Naive   Pallas
                                                  (ms)     (ms)     (ms)
──────────────────────────────────────────────────────────────────────────────────


Mamba2 standard   B=1 H=32  P=64  N=64  Q=64     0.102    0.087    0.053


Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256    0.073    0.190    0.133


Nemotron          B=1 H=128 P=64  N=128 Q=256    0.323    1.341    0.863


Batched           B=2 H=32  P=64  N=64  Q=64     0.086    0.316    0.129
══════════════════════════════════════════════════════════════════════════════════


In [12]:
print("═" * 82)
print(f"FULL PIPELINE  (CB included; Triton=_bmm+_scan, Pallas/Naive=naive_bmm+scan)")
print(f"{'Config':<45} {'Triton':>8} {'Naive':>8} {'Pallas':>8}")
print(f"{'':45} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8}")
print("─" * 82)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    x, dt, dA, C, B, st = [inp[k] for k in ('x','dt','dA','C','B','states')]

    # Triton full: _bmm_chunk_fwd + _chunk_scan_fwd
    x_t  = to_torch_bf16(x);  C_t  = to_torch_bf16(C);  B_t  = to_torch_bf16(B)
    dt_t = to_torch_f32(dt);  dA_t = to_torch_f32(dA);  st_t = to_torch_f32(st)
    def triton_full():
        cb_t = _bmm_chunk_fwd(C_t, B_t, chunk_size, output_dtype=torch.float32)
        return _chunk_scan_fwd(cb_t, x_t, dt_t, dA_t, C_t, st_t)
    t_tri = bench_triton_amortized(triton_full, (), N_AMORTIZE)

    # Naive full: naive bmm + naive scan
    def naive_full(xj, dtj, daj, cj, bj, sj):
        cb = bmm_naive_jax(cj, bj, chunk_size)
        return chunk_scan_naive(cb, xj, dtj, daj, cj, sj)
    fn_naive_full = make_amortized(naive_full, N_AMORTIZE)
    t_naive = bench_amortized(fn_naive_full, (x,dt,dA,C,B,st), N_AMORTIZE)

    # Pallas full: naive bmm + pallas scan
    BM, BN = _best_block(chunk_size, hdim)
    def pallas_full(xj, dtj, daj, cj, bj, sj):
        cb = bmm_naive_jax(cj, bj, chunk_size)
        return chunk_scan_pallas(cb, xj, dtj, daj, cj, sj, BLOCK_M=BM, BLOCK_N=BN)
    fn_pal_full = make_amortized(pallas_full, N_AMORTIZE)
    t_pal = bench_amortized(fn_pal_full, (x,dt,dA,C,B,st), N_AMORTIZE)

    print(f"{label:<45} {t_tri:>8.3f} {t_naive:>8.3f} {t_pal:>8.3f}")

print("═" * 82)

══════════════════════════════════════════════════════════════════════════════════
FULL PIPELINE  (CB included; Triton=_bmm+_scan, Pallas/Naive=naive_bmm+scan)
Config                                          Triton    Naive   Pallas
                                                  (ms)     (ms)     (ms)
──────────────────────────────────────────────────────────────────────────────────


Mamba2 standard   B=1 H=32  P=64  N=64  Q=64     0.200    0.087    0.049


Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256    0.145    0.186    0.133


Nemotron          B=1 H=128 P=64  N=128 Q=256    0.205    1.364    0.911


Batched           B=2 H=32  P=64  N=64  Q=64     0.151    0.313    0.124
══════════════════════════════════════════════════════════════════════════════════


## Analysis

### Precision vs Triton

The Pallas and naive implementations compute `cb_scaled @ x` in **float32 × float32**,
while Triton does `tl.dot(cb.to(bf16), x)` — casting `cb_scaled` to BF16 first.
This produces ≤1 BF16 ULP difference at typical output magnitudes (~64-128):
at those values, BF16 step = 0.5, so max abs diff ≈ 0.5–1.0 vs Triton.

Our F32 matmul is **more accurate** than Triton in this dimension.

### What dominates the runtime

The scan kernel has two GEMM terms:

| Term | Shape | Notes |
|------|-------|-------|
| Y_off: `C @ states^T` | `(BM, dstate) @ (dstate, BN)` | small dstate (64–128) |
| Y_diag: `cb_scaled @ x` | `(BM, chunk_size) @ (chunk_size, BN)` | loads full Q rows of x |

The Y_diag GEMM dominates: it loads `(chunk_size × BN)` of x for **every** (bch, pm)
kernel instance, even though the causal mask zeros the upper-triangular entries.

### Triton's structural advantage: `K_MAX` early-exit

In Triton: `K_MAX = min((pid_m+1)*BM, chunk_size)` cuts the K-loop at the diagonal.
For pm=0: only 1 BK tile of x is loaded.  
For pm=PM-1: all BK tiles are loaded (full x).  
On average: 50% less x bandwidth for Y_diag.

In Pallas: `pm = pl.program_id(axis=1)` is a **traced** int32. We cannot use it in
`if pm * BM < chunk_size - BM:` — Python `if` on a traced value would require `lax.cond`.
Instead we load full chunk_size x and apply `jnp.where(causal, cb_scaled, 0.0)`.
The zero-multiply ops are cheap but x is still loaded — 2× bandwidth vs Triton on average.

**Impact**: most visible for small chunk_size (64, where only half tiles are fully below
the diagonal). For large chunk_size (256), most pm tiles are already below-diagonal,
so the cost is amortized.

### Triton vs Naive vs Pallas

- **Triton** wins: K_MAX causal early-exit, software pipelining (num_stages=3–4),
  hardware-tuned GEMM tile sizes across 11 autotune configs, BF16 matmul throughput.
- **Naive** materializes `L (B,H,C,Q,Q)` and `CB_h` in DRAM when ratio>1 (GQA),
  but cuBLAS saturates for large shapes.
- **Pallas** avoids intermediate materialization (decay applied in-register),
  uses GQA BlockSpec index_map (no repeat), but pays ~2× x bandwidth vs Triton.

### GQA advantage in Pallas

For ngroups=8, nheads=128 (ratio=16): naive uses `jnp.repeat(CB, 16, axis=2)` which
materializes a 16× expanded tensor in DRAM before the matmul. Pallas accesses CB
directly via the `bcg` index map — no expansion, no extra memory bandwidth.

## Pallas v2: K-Tiled Kernel (IS_CAUSAL + Register Pressure Fix)

### Root cause of Nemotron slowness (v1)

v1 loads `(BLOCK_M=32, chunk_size=256)` of CB per kernel instance:
```
register bytes = BM × Q × 4 = 32 × 256 × 4 = 32 KB
```
This causes severe register spilling. Triton uses a K-loop loading only `(BM, BK)` tiles.

### v2 design: K-tiles-as-refs + Python-level loop unrolling

**Key insight**: In JAX (outside the Pallas kernel), we can do static slices freely.
Pre-slice CB and x into `PK = chunk_size // BLOCK_K` K-tiles before calling Pallas:
```python
CB_tiles[k] = CB_flat[:, :, k*BK:(k+1)*BK]   # (BCG, Q, BK) — static slice, valid in JAX
x_tiles[k]  = x_flat[:, k*BK:(k+1)*BK, :]    # (BCH, BK, P)
```
Pass these as `4×PK` separate refs. Inside the kernel, iterate with a **Python for loop**
(unrolled at trace time → PK independent Triton instruction groups):

```python
for k in range(PK):
    cb_k = all_refs[4*k][0, :, :]    # (BM, BK) — static index into Python tuple ✓
    x_k  = all_refs[4*k+1][0, :, :]  # (BK, BN)
    # IS_CAUSAL: k*BK is a compile-time constant, (pm+1)*BM is runtime
    above_diag = k * BK >= (pm + 1) * BM
    acc += lax.cond(above_diag, skip, compute, (cb_k, x_k, dt_k, dAk_k))
```

### Benefits vs v1

| Property | v1 | v2 |
|----------|----|----|
| CB tile size | BM × Q = 32 KB | BM × BK = 8 KB (4×) |
| x tile size | Q × BN = 8 KB | BK × BN = 2 KB (4×) |
| IS_CAUSAL | jnp.where (loads all) | lax.cond (skips compute) |
| Output buffer | (BCH, Q, P) = 2 MB | (BCH, Q, P) = 2 MB (same!) |
| Grid | (BCH, PM, PN) | (BCH, PM, PN) (same!) |

**No intermediate buffer**: output is directly `(BCH, Q, P)` — no 32MB intermediate, no extra reduction.

### IS_CAUSAL savings for Nemotron (PM=PK=4 when BM=BK=64)

Tile `(pm, k)` is above diagonal when `k*BK >= (pm+1)*BM`:
- pm=0: k=1,2,3 skipped → 3 tiles skipped
- pm=1: k=2,3 skipped → 2 tiles skipped
- pm=2: k=3 skipped → 1 tile skipped
- pm=3: nothing skipped

Total: 6 of 16 K-tiles skipped = **37.5% compute savings**.

In [13]:
def _make_y_diag_kernel(PK, BM, BK, BN):
    """
    Factory for the Y_diag kernel.

    Grid: (BCH, PM, PN)  — same 3D structure as v1.

    Inputs (in order): [CB_tile_0, x_tile_0, dt_tile_0, dAk_tile_0,
                        CB_tile_1, x_tile_1, dt_tile_1, dAk_tile_1, ...  (PK groups)
                        dA_m]
    Output: one (BM, BN) tile of Y_diag accumulated over all K-tiles.

    The K-loop is unrolled at Python trace time (static `for k in range(PK)`).
    This means:
    - CB and x tiles are separate refs (no dynamic slicing inside kernel)
    - lax.cond per K-tile implements IS_CAUSAL without a branch on pm
    - Output buffer is (BCH, Q, P) — no 32MB intermediate!
    """
    def kernel(*all_refs):
        # Layout: [CB0, x0, dt0, dAk0,  CB1, x1, dt1, dAk1,  ...  dA_m,  out_ref]
        # Pallas passes in_refs first, then out_refs — last arg is out_ref.
        # Python-level indexing into all_refs is safe (static int indexing of a tuple).
        pm = pl.program_id(axis=1)
        n_in = 4 * PK + 1          # number of input refs
        dA_m  = all_refs[4 * PK][0, :]  # (BM,) — last input ref
        out_ref = all_refs[n_in]         # output ref

        acc = jnp.zeros((BM, BN), dtype=jnp.float32)

        for k in range(PK):
            # Load K-tile refs — all static integer accesses into the Python tuple
            cb_k  = all_refs[4 * k][0, :, :]      # (BM, BK) f32
            x_k   = all_refs[4 * k + 1][0, :, :]  # (BK, BN) bf16
            dt_k  = all_refs[4 * k + 2][0, :]     # (BK,) f32
            dAk_k = all_refs[4 * k + 3][0, :]     # (BK,) f32

            def compute(carry, _k=k):
                cb_k, x_k, dt_k, dAk_k = carry
                # Intra-chunk decay matrix (BM, BK)
                decay     = jnp.exp(jnp.minimum(dA_m[:, None] - dAk_k[None, :], 0.0))
                cb_scaled = cb_k * decay * dt_k[None, :]
                # Causal mask: m_abs (static k → k_abs is a compile-time constant)
                m_abs  = pm * BM + jnp.arange(BM)
                k_abs  = _k * BK + jnp.arange(BK)   # _k is Python int → constant!
                causal = m_abs[:, None] >= k_abs[None, :]
                cb_scaled = jnp.where(causal, cb_scaled, 0.0)
                return jnp.matmul(cb_scaled, x_k.astype(jnp.float32))  # (BM, BN)

            def skip(_):
                return jnp.zeros((BM, BN), dtype=jnp.float32)

            # IS_CAUSAL: skip this K-tile if all row positions m < all col positions k
            # max(m) = (pm+1)*BM - 1 < min(k) = k*BK  ↔  pk*BK >= (pm+1)*BM
            # k is a Python int, so k*BK is a compile-time constant.
            above_diag = k * BK >= (pm + 1) * BM
            acc = acc + lax.cond(above_diag, skip, compute, (cb_k, x_k, dt_k, dAk_k))

        out_ref[0, :, :] = acc

    return kernel


def chunk_scan_pallas_v2(CB, x, dt, dA_cumsum, C, states,
                         BLOCK_M=64, BLOCK_K=64, BLOCK_N=64):
    """
    K-tiled Pallas kernel with IS_CAUSAL and register-local accumulation.

    Key design changes vs v1:
    1. Pre-slice CB, x, dt, dA into PK K-tiles OUTSIDE the kernel (static JAX slices).
    2. Pass 4*PK separate refs to the kernel — Python-level K-loop is unrolled at trace time.
    3. IS_CAUSAL via lax.cond per K-tile: k*BK (compile-time) vs (pm+1)*BM (runtime).
    4. Output is (BCH, Q, P) — no 32MB intermediate buffer, no extra reduction pass.
    5. Y_off computed via pure JAX matmul (as before).
    """
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    BCH = batch * nchunks * nheads
    BCG = batch * nchunks * ngroups
    PM  = chunk_size // BLOCK_M
    PK  = chunk_size // BLOCK_K
    PN  = hdim // BLOCK_N

    # ── Pre-flatten inputs ────────────────────────────────────────────────────
    x_flat = (
        x.reshape(batch, nchunks, chunk_size, nheads, hdim)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCH, chunk_size, hdim)
    )
    CB_flat = CB.reshape(BCG, chunk_size, chunk_size)
    C_flat = (
        C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCG, chunk_size, dstate)
    )
    dt_flat     = dt.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    dA_flat     = dA_cumsum.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    states_flat = states.reshape(BCH, hdim, dstate)

    # ── Y_off: inter-chunk via pure JAX matmul ────────────────────────────────
    bcg_idx  = (jnp.arange(BCH) // nheads) * ngroups + (jnp.arange(BCH) % nheads) // ratio
    C_bch    = C_flat[bcg_idx]                                    # (BCH, Q, N)
    C_scaled = C_bch.astype(jnp.float32) * jnp.exp(dA_flat)[..., None]
    Y_off    = jnp.matmul(C_scaled,
                          states_flat.transpose(0, 2, 1).astype(jnp.float32))  # (BCH, Q, P)

    # ── Pre-slice K-tiles (static slices — valid in JAX wrapper) ─────────────
    # CB_flat: (BCG, Q, Q) → K-tile k: (BCG, Q, BK)
    CB_tiles  = [CB_flat[:, :,  k*BLOCK_K:(k+1)*BLOCK_K] for k in range(PK)]
    # x_flat:  (BCH, Q, P) → K-tile k: (BCH, BK, P)  (rows are K-positions)
    x_tiles   = [x_flat[:,  k*BLOCK_K:(k+1)*BLOCK_K, :] for k in range(PK)]
    # dt/dA_k: (BCH, Q)   → K-tile k: (BCH, BK)
    dt_tiles  = [dt_flat[:, k*BLOCK_K:(k+1)*BLOCK_K]     for k in range(PK)]
    dAk_tiles = [dA_flat[:, k*BLOCK_K:(k+1)*BLOCK_K]     for k in range(PK)]

    # ── BlockSpecs ────────────────────────────────────────────────────────────
    _nh, _ng, _r    = nheads, ngroups, ratio
    _BM, _BK, _BN   = BLOCK_M, BLOCK_K, BLOCK_N

    def _bcg(bch, pm, pn):
        return (bch // _nh) * _ng + (bch % _nh) // _r

    in_specs = []
    for k in range(PK):
        # CB K-tile k: (BCG, Q, BK) block (1, BM, BK) at (bcg, pm, 0)
        in_specs.append(pl.BlockSpec((1, _BM, _BK),
                                     lambda bch, pm, pn: (_bcg(bch, pm, pn), pm, 0)))
        # x K-tile k: (BCH, BK, P) block (1, BK, BN) at (bch, 0, pn)
        in_specs.append(pl.BlockSpec((1, _BK, _BN),
                                     lambda bch, pm, pn: (bch, 0, pn)))
        # dt K-tile k: (BCH, BK) block (1, BK) at (bch, 0)
        in_specs.append(pl.BlockSpec((1, _BK),
                                     lambda bch, pm, pn: (bch, 0)))
        # dAk K-tile k: (BCH, BK) block (1, BK) at (bch, 0)
        in_specs.append(pl.BlockSpec((1, _BK),
                                     lambda bch, pm, pn: (bch, 0)))
    # dA_m: (BCH, Q) block (1, BM) at (bch, pm) — m-row cumsum for IS_CAUSAL condition
    in_specs.append(pl.BlockSpec((1, _BM), lambda bch, pm, pn: (bch, pm)))

    out_spec = [pl.BlockSpec((1, _BM, _BN), lambda bch, pm, pn: (bch, pm, pn))]

    # Build arg list: [CB0, x0, dt0, dAk0,  CB1, x1, dt1, dAk1, ...,  dA_m]
    k_tile_args = []
    for k in range(PK):
        k_tile_args.extend([CB_tiles[k], x_tiles[k], dt_tiles[k], dAk_tiles[k]])
    k_tile_args.append(dA_flat)   # dA_m — last input

    Y_diag, = pl.pallas_call(
        _make_y_diag_kernel(PK, BLOCK_M, BLOCK_K, BLOCK_N),
        out_shape=[jax.ShapeDtypeStruct((BCH, chunk_size, hdim), jnp.float32)],
        in_specs=in_specs,
        out_specs=out_spec,
        grid=(BCH, PM, PN),
        compiler_params=CompilerParams(),
    )(*k_tile_args)

    # Combine Y_off + Y_diag → (batch, seqlen, nheads, hdim)
    Y = (Y_off + Y_diag).reshape(batch, nchunks, nheads, chunk_size, hdim)
    Y = Y.transpose(0, 1, 3, 2, 4).reshape(batch, seqlen, nheads, hdim)
    return Y.astype(jnp.bfloat16)

print("chunk_scan_pallas_v2 (redesigned: K-tiles-as-refs, no intermediate buffer) defined")

chunk_scan_pallas_v2 (redesigned: K-tiles-as-refs, no intermediate buffer) defined


In [14]:
# ── Correctness: standard config (Q=64, ngroups=1) ────────────────────────────
print("=== Standard config (B=2, H=8, Q=64, ngroups=1) ===")
batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = 2, 512, 8, 64, 64, 1, 64
inp = make_inputs(batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size)
CB, x, dt, dA, C, B, states = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

Y_ref_std = jnp.array(
    _chunk_scan_fwd(to_torch_f32(CB), to_torch_bf16(x), to_torch_f32(dt),
                    to_torch_f32(dA), to_torch_bf16(C), to_torch_f32(states))[0]
    .float().cpu().numpy()
).astype(jnp.bfloat16)

Y_v2_std = jax.jit(
    lambda *a: chunk_scan_pallas_v2(*a, BLOCK_M=32, BLOCK_K=32, BLOCK_N=64)
)(CB, x, dt, dA, C, states)
Y_v1_std = jax.jit(
    lambda *a: chunk_scan_pallas(*a, BLOCK_M=32, BLOCK_N=64)
)(CB, x, dt, dA, C, states)

d_v2_tri = float(jnp.max(jnp.abs(Y_v2_std.astype(jnp.float32) - Y_ref_std.astype(jnp.float32))))
d_v1_v2  = float(jnp.max(jnp.abs(Y_v1_std.astype(jnp.float32) - Y_v2_std.astype(jnp.float32))))
print(f"  v2 vs Triton: {d_v2_tri:.4f}   (threshold 2.0)")
print(f"  v1 vs v2:     {d_v1_v2:.4f}   (should be <0.5)")
assert d_v2_tri < 2.0, f"v2 vs Triton too large: {d_v2_tri}"
assert d_v1_v2  < 0.5, f"v1 vs v2 too large: {d_v1_v2}"
print("  OK")

# ── Correctness: Nemotron config (Q=256, ngroups=8) ───────────────────────────
print("\n=== Nemotron config (B=1, H=128, Q=256, ngroups=8) ===")
inp2 = make_inputs(1, 256, 128, 64, 128, 8, 256)
CB2, x2, dt2, dA2, C2, B2, st2 = [inp2[k] for k in ('CB','x','dt','dA','C','B','states')]

Y_ref2 = jnp.array(
    _chunk_scan_fwd(to_torch_f32(CB2), to_torch_bf16(x2), to_torch_f32(dt2),
                    to_torch_f32(dA2), to_torch_bf16(C2), to_torch_f32(st2))[0]
    .float().cpu().numpy()
).astype(jnp.bfloat16)

Y_v2_nem = jax.jit(
    lambda *a: chunk_scan_pallas_v2(*a, BLOCK_M=64, BLOCK_K=64, BLOCK_N=64)
)(CB2, x2, dt2, dA2, C2, st2)
Y_v1_nem = jax.jit(
    lambda *a: chunk_scan_pallas(*a, BLOCK_M=64, BLOCK_N=64)
)(CB2, x2, dt2, dA2, C2, st2)

d_v2_tri2 = float(jnp.max(jnp.abs(Y_v2_nem.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
d_v1_v22  = float(jnp.max(jnp.abs(Y_v1_nem.astype(jnp.float32) - Y_v2_nem.astype(jnp.float32))))
print(f"  v2 vs Triton: {d_v2_tri2:.4f}   (threshold 2.0)")
print(f"  v1 vs v2:     {d_v1_v22:.4f}   (should be <0.5)")
assert d_v2_tri2 < 2.0, f"v2 vs Triton too large: {d_v2_tri2}"
assert d_v1_v22  < 0.5, f"v1 vs v2 too large: {d_v1_v22}"
print("  OK")

=== Standard config (B=2, H=8, Q=64, ngroups=1) ===


  v2 vs Triton: 0.5000   (threshold 2.0)
  v1 vs v2:     0.2500   (should be <0.5)
  OK

=== Nemotron config (B=1, H=128, Q=256, ngroups=8) ===


  v2 vs Triton: 1.0000   (threshold 2.0)
  v1 vs v2:     0.2500   (should be <0.5)
  OK


In [15]:
# Autotune v2 on Nemotron config (the critical shape to optimize)
_nem_cfg = dict(batch=1, seqlen=256, nheads=128, hdim=64, dstate=128, ngroups=8, chunk_size=256)
_nem_inp = make_inputs(**_nem_cfg)
_CB2, _x2, _dt2, _dA2, _C2, _st2 = [_nem_inp[k] for k in ('CB','x','dt','dA','C','states')]

print(f"Autotuning v2 for Nemotron: {_nem_cfg}")
print(f"  {'BM':>4} {'BK':>4} {'BN':>4}     ms")
print("-" * 36)

BEST_V2_BM, BEST_V2_BK, BEST_V2_BN = 64, 64, 64
best_v2_ms = float('inf')

for BM in [32, 64]:
    for BK in [32, 64]:
        for BN in [32, 64]:
            cs = _nem_cfg['chunk_size']; hd = _nem_cfg['hdim']
            if cs % BM != 0 or cs % BK != 0 or hd % BN != 0:
                continue
            fn = jax.jit(
                lambda cb,x,dt,da,c,st,BM=BM,BK=BK,BN=BN:
                    chunk_scan_pallas_v2(cb,x,dt,da,c,st,BLOCK_M=BM,BLOCK_K=BK,BLOCK_N=BN)
            )
            for _ in range(3): fn(_CB2,_x2,_dt2,_dA2,_C2,_st2).block_until_ready()
            t0 = time.perf_counter()
            for _ in range(20): fn(_CB2,_x2,_dt2,_dA2,_C2,_st2).block_until_ready()
            ms = (time.perf_counter()-t0)/20*1e3
            tag = "  <-- best" if ms < best_v2_ms else ""
            print(f"  {BM:4d} {BK:4d} {BN:4d}  {ms:6.3f} ms{tag}")
            if ms < best_v2_ms:
                best_v2_ms, BEST_V2_BM, BEST_V2_BK, BEST_V2_BN = ms, BM, BK, BN

print(f"\nBest v2: BLOCK_M={BEST_V2_BM}, BLOCK_K={BEST_V2_BK}, BLOCK_N={BEST_V2_BN}")

Autotuning v2 for Nemotron: {'batch': 1, 'seqlen': 256, 'nheads': 128, 'hdim': 64, 'dstate': 128, 'ngroups': 8, 'chunk_size': 256}
    BM   BK   BN     ms
------------------------------------


    32   32   32   1.152 ms  <-- best


    32   32   64   1.148 ms  <-- best


    32   64   32   1.143 ms  <-- best


    32   64   64   1.147 ms


    64   32   32   1.303 ms


    64   32   64   1.365 ms


    64   64   32   1.160 ms


    64   64   64   1.260 ms

Best v2: BLOCK_M=32, BLOCK_K=64, BLOCK_N=32


In [16]:
def _best_block_v2(chunk_size, hdim):
    """Pick best v2 tile sizes: prefer (64, 64, 64) when shape allows."""
    bm = max(b for b in [32, 64] if chunk_size % b == 0)
    bk = max(b for b in [32, 64] if chunk_size % b == 0)
    bn = max(b for b in [32, 64] if hdim % b == 0)
    return bm, bk, bn

print("═" * 90)
print("SCAN-ONLY: v1 vs v2 vs Triton  (CB pre-computed, not counted)")
print(f"{'Config':<45} {'Triton':>8} {'v1':>8} {'v2':>8} {'v2/Tri':>8}")
print(f"{'':45} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8} {'speedup':>8}")
print("─" * 90)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    CB, x, dt, dA, C, B, st = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

    # Triton
    CB_t = to_torch_f32(CB); x_t = to_torch_bf16(x)
    dt_t = to_torch_f32(dt); dA_t = to_torch_f32(dA)
    C_t  = to_torch_bf16(C); st_t = to_torch_f32(st)
    t_tri = bench_triton_amortized(
        lambda: _chunk_scan_fwd(CB_t, x_t, dt_t, dA_t, C_t, st_t), (), N_AMORTIZE
    )

    # v1 Pallas
    BM, BN = _best_block(chunk_size, hdim)
    fn_v1 = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj: chunk_scan_pallas(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_N=BN),
        N_AMORTIZE
    )
    t_v1 = bench_amortized(fn_v1, (CB,x,dt,dA,C,st), N_AMORTIZE)

    # v2 Pallas
    BM2, BK2, BN2 = _best_block_v2(chunk_size, hdim)
    fn_v2 = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj: chunk_scan_pallas_v2(cb,xj,dtj,daj,cj,sj,
                                                           BLOCK_M=BM2,BLOCK_K=BK2,BLOCK_N=BN2),
        N_AMORTIZE
    )
    t_v2 = bench_amortized(fn_v2, (CB,x,dt,dA,C,st), N_AMORTIZE)

    ratio_str = f"{t_tri/t_v2:.2f}x" if t_v2 > 0 else "?"
    print(f"{label:<45} {t_tri:>8.3f} {t_v1:>8.3f} {t_v2:>8.3f} {ratio_str:>8}")

print("═" * 90)

print()
print("═" * 90)
print("FULL PIPELINE: v1 vs v2 vs Triton  (CB included)")
print(f"{'Config':<45} {'Triton':>8} {'v1':>8} {'v2':>8} {'v2/Tri':>8}")
print(f"{'':45} {'(ms)':>8} {'(ms)':>8} {'(ms)':>8} {'speedup':>8}")
print("─" * 90)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    x, dt, dA, C, B, st = [inp[k] for k in ('x','dt','dA','C','B','states')]

    x_t  = to_torch_bf16(x);  C_t  = to_torch_bf16(C);  B_t  = to_torch_bf16(B)
    dt_t = to_torch_f32(dt);  dA_t = to_torch_f32(dA);  st_t = to_torch_f32(st)
    def triton_full():
        cb_t = _bmm_chunk_fwd(C_t, B_t, chunk_size, output_dtype=torch.float32)
        return _chunk_scan_fwd(cb_t, x_t, dt_t, dA_t, C_t, st_t)
    t_tri = bench_triton_amortized(triton_full, (), N_AMORTIZE)

    BM, BN = _best_block(chunk_size, hdim)
    def v1_full(xj,dtj,daj,cj,bj,sj):
        cb = bmm_naive_jax(cj, bj, chunk_size)
        return chunk_scan_pallas(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_N=BN)
    t_v1 = bench_amortized(make_amortized(v1_full, N_AMORTIZE), (x,dt,dA,C,B,st), N_AMORTIZE)

    BM2, BK2, BN2 = _best_block_v2(chunk_size, hdim)
    def v2_full(xj,dtj,daj,cj,bj,sj):
        cb = bmm_naive_jax(cj, bj, chunk_size)
        return chunk_scan_pallas_v2(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM2,BLOCK_K=BK2,BLOCK_N=BN2)
    t_v2 = bench_amortized(make_amortized(v2_full, N_AMORTIZE), (x,dt,dA,C,B,st), N_AMORTIZE)

    ratio_str = f"{t_tri/t_v2:.2f}x" if t_v2 > 0 else "?"
    print(f"{label:<45} {t_tri:>8.3f} {t_v1:>8.3f} {t_v2:>8.3f} {ratio_str:>8}")

print("═" * 90)

══════════════════════════════════════════════════════════════════════════════════════════
SCAN-ONLY: v1 vs v2 vs Triton  (CB pre-computed, not counted)
Config                                          Triton       v1       v2   v2/Tri
                                                  (ms)     (ms)     (ms)  speedup
──────────────────────────────────────────────────────────────────────────────────────────


Mamba2 standard   B=1 H=32  P=64  N=64  Q=64     0.086    0.147    0.053    1.63x


Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256    0.094    0.132    0.073    1.29x


Nemotron          B=1 H=128 P=64  N=128 Q=256    0.191    0.863    0.798    0.24x


Batched           B=2 H=32  P=64  N=64  Q=64     0.110    0.129    0.208    0.53x
══════════════════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════════════════
FULL PIPELINE: v1 vs v2 vs Triton  (CB included)
Config                                          Triton       v1       v2   v2/Tri
                                                  (ms)     (ms)     (ms)  speedup
──────────────────────────────────────────────────────────────────────────────────────────


Mamba2 standard   B=1 H=32  P=64  N=64  Q=64     0.136    0.049    0.049    2.76x


Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256    0.117    0.136    0.082    1.41x


Nemotron          B=1 H=128 P=64  N=128 Q=256    0.206    0.909    0.832    0.25x


Batched           B=2 H=32  P=64  N=64  Q=64     0.135    0.125    0.209    0.65x
══════════════════════════════════════════════════════════════════════════════════════════


## Analysis: v2 Results

### What v2 improves

For small-Q configs (Q=64) and moderate ngroups (ngroups=1), v2 is **faster than Triton**:
- Standard Q=64: v2=0.054ms vs Triton=0.086ms → **1.6× faster**
- cs=256 (Q=256, ngroups=1): v2=0.073ms vs Triton=0.089ms → **1.2× faster**

The improvement comes from:
1. Smaller CB tiles (BM×BK instead of BM×Q) → less register pressure, better SM occupancy
2. IS_CAUSAL via `lax.cond`: K-tiles for k > pm are skipped (compute avoided)
3. No intermediate buffer overhead

### Why Nemotron (Q=256, ngroups=8) is still slow

v2 for Nemotron: **0.795ms** vs Triton: **0.185ms** (v2 ≈ v1 = 0.776ms, no improvement)

**Root cause: IS_CAUSAL doesn't save bandwidth in v2**

In the current v2, ref loads happen BEFORE `lax.cond`:
```python
cb_k = all_refs[4*k][0, :, :]    # ← load happens HERE (before lax.cond)
...
acc += lax.cond(above_diag, skip, compute, (cb_k, x_k, ...))  # skip doesn't un-load
```
For pm=0 with PK=4: 3 of 4 K-tiles are above-diagonal, but ALL 4 are loaded from HBM.  
Total loaded for Nemotron (BCH=1024, PM=4, PK=4): **~400 MB** vs Triton's ~150 MB.

**Triton's advantages for large Q**:
1. True IS_CAUSAL: `for k in range(0, K_MAX, BK)` — loader and compute only run for active tiles
2. Software pipelining (`num_stages=3-4`): next K-tile's loads overlap with current K-tile's compute
3. BF16 matmul: `tl.dot(cb.to(bf16), x)` uses tensor cores at 165 TFLOPs vs our FP32 at 82 TFLOPs

### Potential next step: true IS_CAUSAL via loads inside `lax.cond`

Moving loads INSIDE the compute branch would enable conditional loading:
```python
def compute(_, _k=k):
    cb_k = all_refs[4*_k][0, :, :]   # load only if branch is taken
    x_k  = all_refs[4*_k+1][0, :, :]
    # ...

acc += lax.cond(above_diag, skip, compute, None)   # load only when compute branch runs
```
Whether Pallas/Triton lowers conditional ref access to actual conditional loads
is an open question (depends on XLA's dead-code elimination of not-taken branches).

### Summary table

| Config | Triton | v1 | v2 | v2/Triton |
|--------|--------|----|----|-----------|
| Q=64, ngroups=1 | 0.086ms | 0.126ms | **0.054ms** | **1.6× faster** |
| Q=256, ngroups=1 | 0.089ms | 0.153ms | **0.073ms** | **1.2× faster** |
| Q=256, ngroups=8 (Nemotron) | 0.185ms | 0.776ms | 0.795ms | 4.3× slower |
| B=2, Q=64 | 0.069ms | 0.146ms | 0.222ms | 3.2× slower |

Pallas v2 excels for small-Q configs where IS_CAUSAL is effective and register pressure is the bottleneck.
For large-Q configs with many heads, Triton's IS_CAUSAL K_MAX loop + software pipelining are decisive.

In [17]:
# ── v2b: loads INSIDE lax.cond — test if Pallas/Triton generates conditional loads ──────
# If ref accesses inside lax.cond branches actually avoid the HBM loads for above-diagonal
# tiles, this should significantly help Nemotron (reduces loaded data by ~37.5%).

def _make_y_diag_kernel_v2b(PK, BM, BK, BN):
    """Same as v2 but ref loads are inside the lax.cond compute branch."""
    def kernel(*all_refs):
        pm   = pl.program_id(axis=1)
        dA_m = all_refs[4 * PK][0, :]   # (BM,)
        out_ref = all_refs[4 * PK + 1]

        acc = jnp.zeros((BM, BN), dtype=jnp.float32)

        for k in range(PK):
            def compute(_, _k=k):
                # Loads are INSIDE the compute branch — only executed if above_diag=False
                cb_k  = all_refs[4 * _k][0, :, :]       # (BM, BK) f32
                x_k   = all_refs[4 * _k + 1][0, :, :]   # (BK, BN) bf16
                dt_k  = all_refs[4 * _k + 2][0, :]      # (BK,) f32
                dAk_k = all_refs[4 * _k + 3][0, :]      # (BK,) f32
                decay     = jnp.exp(jnp.minimum(dA_m[:, None] - dAk_k[None, :], 0.0))
                cb_scaled = cb_k * decay * dt_k[None, :]
                m_abs  = pm * BM + jnp.arange(BM)
                k_abs  = _k * BK + jnp.arange(BK)
                causal = m_abs[:, None] >= k_abs[None, :]
                cb_scaled = jnp.where(causal, cb_scaled, 0.0)
                return jnp.matmul(cb_scaled, x_k.astype(jnp.float32))

            def skip(_):
                return jnp.zeros((BM, BN), dtype=jnp.float32)

            above_diag = k * BK >= (pm + 1) * BM
            acc = acc + lax.cond(above_diag, skip, compute, None)

        out_ref[0, :, :] = acc
    return kernel


def chunk_scan_pallas_v2b(CB, x, dt, dA_cumsum, C, states,
                           BLOCK_M=64, BLOCK_K=64, BLOCK_N=64):
    """v2 variant: ref loads inside lax.cond branches (potential IS_CAUSAL bandwidth savings)."""
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    BCH = batch * nchunks * nheads
    BCG = batch * nchunks * ngroups
    PM  = chunk_size // BLOCK_M
    PK  = chunk_size // BLOCK_K
    PN  = hdim // BLOCK_N

    x_flat = (x.reshape(batch, nchunks, chunk_size, nheads, hdim)
               .transpose(0, 1, 3, 2, 4).reshape(BCH, chunk_size, hdim))
    CB_flat = CB.reshape(BCG, chunk_size, chunk_size)
    C_flat  = (C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
                .transpose(0, 1, 3, 2, 4).reshape(BCG, chunk_size, dstate))
    dt_flat     = dt.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    dA_flat     = dA_cumsum.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    states_flat = states.reshape(BCH, hdim, dstate)

    bcg_idx  = (jnp.arange(BCH) // nheads) * ngroups + (jnp.arange(BCH) % nheads) // ratio
    C_bch    = C_flat[bcg_idx]
    C_scaled = C_bch.astype(jnp.float32) * jnp.exp(dA_flat)[..., None]
    Y_off    = jnp.matmul(C_scaled, states_flat.transpose(0, 2, 1).astype(jnp.float32))

    CB_tiles  = [CB_flat[:, :,  k*BLOCK_K:(k+1)*BLOCK_K] for k in range(PK)]
    x_tiles   = [x_flat[:,  k*BLOCK_K:(k+1)*BLOCK_K, :]  for k in range(PK)]
    dt_tiles  = [dt_flat[:, k*BLOCK_K:(k+1)*BLOCK_K]      for k in range(PK)]
    dAk_tiles = [dA_flat[:, k*BLOCK_K:(k+1)*BLOCK_K]      for k in range(PK)]

    _nh, _ng, _r  = nheads, ngroups, ratio
    _BM, _BK, _BN = BLOCK_M, BLOCK_K, BLOCK_N

    def _bcg(bch, pm, pn): return (bch // _nh) * _ng + (bch % _nh) // _r

    in_specs = []
    for k in range(PK):
        in_specs.append(pl.BlockSpec((1, _BM, _BK),
                                     lambda bch, pm, pn: (_bcg(bch, pm, pn), pm, 0)))
        in_specs.append(pl.BlockSpec((1, _BK, _BN),
                                     lambda bch, pm, pn: (bch, 0, pn)))
        in_specs.append(pl.BlockSpec((1, _BK), lambda bch, pm, pn: (bch, 0)))
        in_specs.append(pl.BlockSpec((1, _BK), lambda bch, pm, pn: (bch, 0)))
    in_specs.append(pl.BlockSpec((1, _BM), lambda bch, pm, pn: (bch, pm)))

    out_spec = [pl.BlockSpec((1, _BM, _BN), lambda bch, pm, pn: (bch, pm, pn))]

    k_tile_args = []
    for k in range(PK):
        k_tile_args.extend([CB_tiles[k], x_tiles[k], dt_tiles[k], dAk_tiles[k]])
    k_tile_args.append(dA_flat)

    Y_diag, = pl.pallas_call(
        _make_y_diag_kernel_v2b(PK, BLOCK_M, BLOCK_K, BLOCK_N),
        out_shape=[jax.ShapeDtypeStruct((BCH, chunk_size, hdim), jnp.float32)],
        in_specs=in_specs,
        out_specs=out_spec,
        grid=(BCH, PM, PN),
        compiler_params=CompilerParams(),
    )(*k_tile_args)

    Y = (Y_off + Y_diag).reshape(batch, nchunks, nheads, chunk_size, hdim)
    Y = Y.transpose(0, 1, 3, 2, 4).reshape(batch, seqlen, nheads, hdim)
    return Y.astype(jnp.bfloat16)

print("chunk_scan_pallas_v2b (loads inside lax.cond) defined")

chunk_scan_pallas_v2b (loads inside lax.cond) defined


In [18]:
# Test v2b correctness and Nemotron benchmark vs v2
print("=== v2b correctness (Nemotron) ===")
inp2 = make_inputs(1, 256, 128, 64, 128, 8, 256)
CB2, x2, dt2, dA2, C2, B2, st2 = [inp2[k] for k in ('CB','x','dt','dA','C','B','states')]
Y_ref2 = jnp.array(
    _chunk_scan_fwd(to_torch_f32(CB2), to_torch_bf16(x2), to_torch_f32(dt2),
                    to_torch_f32(dA2), to_torch_bf16(C2), to_torch_f32(st2))[0]
    .float().cpu().numpy()
).astype(jnp.bfloat16)

try:
    Y_v2b = jax.jit(
        lambda *a: chunk_scan_pallas_v2b(*a, BLOCK_M=64, BLOCK_K=64, BLOCK_N=64)
    )(CB2, x2, dt2, dA2, C2, st2)
    d = float(jnp.max(jnp.abs(Y_v2b.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
    print(f"  v2b vs Triton: {d:.4f}  (threshold 2.0)")
    assert d < 2.0
    print("  OK — v2b is correct")

    # Benchmark v2b vs v2 for Nemotron
    print("\n=== Nemotron benchmark: v2 vs v2b (loads inside lax.cond) ===")
    inp_n = make_inputs(1, 2048, 128, 64, 128, 8, 256)
    CB_n, x_n, dt_n, dA_n, C_n, st_n = [inp_n[k] for k in ('CB','x','dt','dA','C','states')]

    BM2, BK2, BN2 = 64, 64, 64
    fn_v2_n  = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj: chunk_scan_pallas_v2(*[cb,xj,dtj,daj,cj,sj],
                                                           BLOCK_M=BM2,BLOCK_K=BK2,BLOCK_N=BN2),
        N_AMORTIZE)
    fn_v2b_n = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj: chunk_scan_pallas_v2b(*[cb,xj,dtj,daj,cj,sj],
                                                            BLOCK_M=BM2,BLOCK_K=BK2,BLOCK_N=BN2),
        N_AMORTIZE)

    t_v2_n  = bench_amortized(fn_v2_n,  (CB_n,x_n,dt_n,dA_n,C_n,st_n), N_AMORTIZE)
    t_v2b_n = bench_amortized(fn_v2b_n, (CB_n,x_n,dt_n,dA_n,C_n,st_n), N_AMORTIZE)

    CB_t_n = to_torch_f32(CB_n); x_t_n = to_torch_bf16(x_n)
    dt_t_n = to_torch_f32(dt_n); dA_t_n = to_torch_f32(dA_n)
    C_t_n  = to_torch_bf16(C_n); st_t_n = to_torch_f32(st_n)
    t_tri_n = bench_triton_amortized(
        lambda: _chunk_scan_fwd(CB_t_n, x_t_n, dt_t_n, dA_t_n, C_t_n, st_t_n), (), N_AMORTIZE)

    print(f"  Triton:          {t_tri_n:.3f} ms")
    print(f"  v2  (loads out): {t_v2_n:.3f} ms")
    print(f"  v2b (loads in):  {t_v2b_n:.3f} ms")
    if t_v2b_n < t_v2_n:
        print(f"  → v2b is {t_v2_n/t_v2b_n:.2f}× faster (conditional loads DO save bandwidth!)")
    else:
        print(f"  → No improvement (conditional loads NOT eliminating HBM bandwidth)")

except Exception as e:
    print(f"  ERROR: {e}")
    print("  v2b failed — ref accesses inside lax.cond branches may not be supported")

=== v2b correctness (Nemotron) ===


  v2b vs Triton: 1.0000  (threshold 2.0)
  OK — v2b is correct

=== Nemotron benchmark: v2 vs v2b (loads inside lax.cond) ===


  Triton:          0.188 ms
  v2  (loads out): 0.796 ms
  v2b (loads in):  0.792 ms
  → v2b is 1.00× faster (conditional loads DO save bandwidth!)



## Pallas v3: Per-Row-Chunk Dispatch (True Triangular Bandwidth Savings)

### Why v2's IS_CAUSAL doesn't save HBM bandwidth

v2 passes **all PK K-tile refs** to every `(BCH, PM, PN)` kernel instance.
Pallas/Triton issues unconditional `tl.load` for every ref regardless of branch
predicates — confirmed by v2b experiment (loads moved inside `lax.cond`: 0% improvement).

For Nemotron (PM=PK=4, BM=BK=64):
```
pm=0: loads 4 CB tiles, uses 1  → 75% wasted HBM bandwidth
pm=1: loads 4 CB tiles, uses 2  → 50% wasted
pm=2: loads 4 CB tiles, uses 3  → 25% wasted
pm=3: loads 4 CB tiles, uses 4  → 0% wasted
```
Total: 16 tile-loads vs 10 useful → **37.5% wasted bandwidth**.

### v3: PM separate `pallas_call` invocations

Loop outside the kernel: `for pm_chunk in range(PM):`

Since `pm_chunk` is a Python int (not a traced `pl.program_id` value), we can compute:
```python
K_active = ceil((pm_chunk + 1) * BM / BK)   # Python int — static per invocation
```
and pre-slice CB to exactly those K-tiles before calling Pallas:
```python
CB_row_tiles = [CB_flat[:, pm_chunk*BM:(pm_chunk+1)*BM, k*BK:(k+1)*BK]
                for k in range(K_active)]    # STATIC JAX slices — K_active refs only
```

Each `pallas_call` receives K_active refs. Above-diagonal tiles **don't exist as refs** →
cannot be loaded by any mechanism. Savings are structural, not conditional.

```
pm_chunk=0: K_active=1 ref passed  → kernel loads 1 CB tile
pm_chunk=1: K_active=2 refs passed → kernel loads 2 CB tiles
pm_chunk=2: K_active=3 refs passed → kernel loads 3 CB tiles
pm_chunk=3: K_active=4 refs passed → kernel loads 4 CB tiles
Total: 10 loads vs v2's 16  →  37.5% savings  (guaranteed by construction)
```

### Why the outer loop cannot be inside the kernel

If `pm` came from `pl.program_id(axis=1)` (traced), then:
- `K_active = pm + 1` → traced scalar → `range(K_active)` → illegal (needs static bound)
- Workaround via `lax.fori_loop` → body must index refs dynamically → `lax.slice NotImplementedError`

So `pm_chunk` must be a Python int ↔ loop at wrapper level.

### Why triangular sparse grid doesn't work

A grid of active `(pm, k)` pairs would have multiple kernel instances (different k, same pm)
needing to accumulate into the **same output block** `Y[bch, pm, pn]`.
Pallas `BlockSpec` gives exclusive output ownership per instance — no atomic-add across blocks.
Fixing this requires a 160MB intermediate buffer (worse than the 32MB we already rejected).

### Static causal mask

Since both `pm_chunk` and `k` are Python ints, the mask for the diagonal tile is a
**compile-time constant matrix** baked into the generated PTX — zero runtime overhead:
```python
m_abs  = pm_chunk * BM + jnp.arange(BM)   # constant array
k_abs  = k * BK       + jnp.arange(BK)    # constant array
causal = m_abs[:, None] >= k_abs[None, :]  # constant bool matrix → PTX immediates
```
For fully-causal tiles (`pm_chunk*BM >= (k+1)*BK - 1`): Python `if` skips the mask
entirely — no `jnp.where` emitted at all.


In [19]:

def _make_y_diag_kernel_v3(pm_chunk, K_active, BM, BK, BN):
    """
    Factory: generates the Y_diag kernel for ONE row-chunk pm_chunk.

    All of pm_chunk, K_active, BM, BK, BN are Python ints baked into the kernel
    at trace time, so:
      - `for k in range(K_active)` is a Python loop → unrolled into K_active
        independent Triton instruction groups (no dynamic indexing).
      - `if pm_chunk * BM < (k+1)*BK - 1` is a Python if → evaluated at trace time,
        so the causal mask either IS or IS NOT emitted as PTX instructions per k.
      - `m_abs = pm_chunk*BM + jnp.arange(BM)` is a compile-time constant array
        (pm_chunk is an int, arange is static) → mask comparison baked as immediates.
      - No lax.cond needed: above-diagonal tiles are never passed as refs, so they
        cannot be loaded. The kernel handles only K_active ≤ PK tiles.

    Grid: (BCH, PN) — 2D.  pm is gone from the grid (fixed to pm_chunk).

    Inputs (in order per K-tile group k = 0..K_active-1):
        CB_tile_k  : (BCG, BM, BK)  float32  — row already pre-sliced to pm_chunk
        x_tile_k   : (BCH, BK, BN)  bfloat16
        dt_tile_k  : (BCH, BK)      float32
        dAk_tile_k : (BCH, BK)      float32  — cumsum at K positions
    Followed by:
        dA_m       : (BCH, BM)      float32  — cumsum at M positions for this row-chunk
    Output:
        out        : (BCH, BM, BN)  float32  — accumulated Y_diag for this row-chunk
    """
    def kernel(*all_refs):
        # dA_m is the last input ref (index 4*K_active); out_ref follows it
        dA_m    = all_refs[4 * K_active][0, :]   # (BM,)
        out_ref = all_refs[4 * K_active + 1]      # output ref, block shape (1, BM, BN)

        acc = jnp.zeros((BM, BN), dtype=jnp.float32)

        for k in range(K_active):   # Python for → unrolled; K_active is a Python int
            # Static integer indexing into the all_refs tuple (safe in Pallas/Triton)
            cb_k  = all_refs[4 * k][0, :, :]      # (BM, BK) float32
            x_k   = all_refs[4 * k + 1][0, :, :]  # (BK, BN) bfloat16
            dt_k  = all_refs[4 * k + 2][0, :]     # (BK,)    float32
            dAk_k = all_refs[4 * k + 3][0, :]     # (BK,)    float32 — cumsum at K positions

            # ── Intra-chunk decay matrix: (BM, BK) ────────────────────────────
            # decay[i, j] = exp(dA_m[i] - dAk_k[j])  clamped to ≤ 0
            # The clamp prevents exp overflow: dA is a cumulative sum of log-decays
            # (non-negative values), so for m < k we'd get large positive differences.
            # Clamping to 0 means those entries decay to at most 1.0 (handled by
            # the causal mask below, which zeros them out anyway).
            decay     = jnp.exp(jnp.minimum(dA_m[:, None] - dAk_k[None, :], 0.0))  # (BM, BK)
            cb_scaled = cb_k * decay * dt_k[None, :]                                 # (BM, BK)

            # ── Causal mask (Python if → compile-time decision) ───────────────
            # Tile (pm_chunk, k) is FULLY CAUSAL when all M rows are ≥ all K cols:
            #   min(m_abs) >= max(k_abs)
            #   pm_chunk*BM  >=  k*BK + BK - 1  =  (k+1)*BK - 1
            # If this holds: skip the mask (all elements valid, no zeroing needed).
            # If this does NOT hold: some K positions exceed some M positions → mask.
            if pm_chunk * BM < (k + 1) * BK - 1:
                # Diagonal or partially-above-diagonal tile: zero out strictly-above entries.
                # m_abs and k_abs are compile-time constants (pm_chunk, k are Python ints).
                m_abs  = pm_chunk * BM + jnp.arange(BM)   # (BM,) — absolute M positions
                k_abs  = k * BK       + jnp.arange(BK)    # (BK,) — absolute K positions
                causal = m_abs[:, None] >= k_abs[None, :]  # (BM, BK) bool, baked into PTX
                cb_scaled = jnp.where(causal, cb_scaled, 0.0)
            # else: pm_chunk*BM >= (k+1)*BK - 1 → fully below diagonal, no mask needed

            # Accumulate: (BM, BK) @ (BK, BN) → (BM, BN) in float32
            acc = acc + jnp.matmul(cb_scaled, x_k.astype(jnp.float32))

        out_ref[0, :, :] = acc

    return kernel


def chunk_scan_pallas_v3(CB, x, dt, dA_cumsum, C, states,
                         BLOCK_M=64, BLOCK_K=64, BLOCK_N=64):
    """
    Per-row-chunk Pallas implementation of _chunk_scan_fwd (Y_diag term).

    Core idea vs v2:
      v2: one pallas_call for the entire (BCH, PM, PN) grid, all PK K-tiles passed
          as refs → Pallas loads ALL PK K-tiles unconditionally for every instance,
          even those above the diagonal (lax.cond skips compute but NOT HBM loads).

      v3: PM separate pallas_calls, one per row-chunk pm_chunk.
          Each call receives K_active = ceil((pm_chunk+1)*BM/BK) K-tile refs — just
          the tiles that lie on or below the diagonal for that row-chunk.
          Above-diagonal tiles are NEVER passed → structurally cannot be loaded.
          Expected 37.5% bandwidth savings for Nemotron (PM=PK=4, BM=BK=64).

    Why the outer loop must be at Python/wrapper level:
      Inside a Pallas kernel, pm would be a traced scalar from pl.program_id.
      K_active = pm + 1 would then be traced → `for k in range(K_active)` would
      require lax.fori_loop with dynamic indexing → lax.slice NotImplementedError.
      At wrapper level pm_chunk is a Python int → K_active is a Python int → safe.

    Tile size trade-off:
      BM = BK (default): square K-tiles, bandwidth savings plateau at ~37.5% for PM=PK.
      BM > BK: each row-chunk needs more K-tiles → savings shrink toward 0 as BM→Q.
      BM < BK: same savings as BM=BK but more pallas_call dispatches & smaller matmuls.
    """
    batch, seqlen, nheads, hdim = x.shape
    _, _, nchunks, chunk_size   = dt.shape
    _, _, ngroups, dstate       = C.shape
    ratio = nheads // ngroups

    BCH = batch * nchunks * nheads   # flattened (batch, chunk, head) axis
    BCG = batch * nchunks * ngroups  # flattened (batch, chunk, group) axis — for CB
    PM  = chunk_size // BLOCK_M      # number of row-chunks in the M (sequence) dimension
    PK  = chunk_size // BLOCK_K      # total K-tiles (used to cap K_active)
    PN  = hdim // BLOCK_N            # tiles along the output N (hdim) dimension

    # ── Pre-flatten inputs (identical layout to v1/v2) ────────────────────────
    # x: (batch, seqlen, nheads, hdim)
    #    → interleave chunk/head → (batch, nchunks, nheads, chunk_size, hdim)
    #    → (BCH, chunk_size, hdim)
    x_flat = (
        x.reshape(batch, nchunks, chunk_size, nheads, hdim)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCH, chunk_size, hdim)
    )
    # CB: (batch, nchunks, ngroups, chunk_size, chunk_size) from bmm step
    CB_flat = CB.reshape(BCG, chunk_size, chunk_size)             # (BCG, Q, Q)

    # C: (batch, seqlen, ngroups, dstate) → (BCG, chunk_size, dstate)
    C_flat = (
        C.reshape(batch, nchunks, chunk_size, ngroups, dstate)
         .transpose(0, 1, 3, 2, 4)
         .reshape(BCG, chunk_size, dstate)
    )
    # dt, dA: (batch, nheads, nchunks, chunk_size) → (BCH, chunk_size)
    dt_flat  = dt.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    dA_flat  = dA_cumsum.transpose(0, 2, 1, 3).reshape(BCH, chunk_size)
    # states: (batch, nheads, nchunks, hdim, dstate) or equivalent
    states_flat = states.reshape(BCH, hdim, dstate)

    # ── Y_off: inter-chunk contribution (pure JAX, same as v1/v2) ─────────────
    # Y_off[bch, t, p] = (C[bch,t] * exp(dA[bch,t])) @ states[bch,:,:]^T
    # GQA: each of the `ratio` heads in a group shares one CB and one states block.
    bcg_idx  = (jnp.arange(BCH) // nheads) * ngroups + (jnp.arange(BCH) % nheads) // ratio
    C_bch    = C_flat[bcg_idx]                                                  # (BCH, Q, N)
    C_scaled = C_bch.astype(jnp.float32) * jnp.exp(dA_flat)[..., None]         # (BCH, Q, N)
    Y_off    = jnp.matmul(C_scaled,
                          states_flat.transpose(0, 2, 1).astype(jnp.float32))  # (BCH, Q, P)

    # ── Per-row-chunk dispatch ────────────────────────────────────────────────
    # BlockSpec index functions for the 2D grid (BCH, PN).
    # All lambdas take (bch, pn); pm_chunk is a Python int (baked into each closure).
    _nh, _ng, _r  = nheads, ngroups, ratio
    _BM, _BK, _BN = BLOCK_M, BLOCK_K, BLOCK_N

    def _bcg(bch, pn):
        # Map flattened head index bch → flattened group index bcg for CB/C lookup.
        # Formula: bcg = (batch_chunk_idx) * ngroups + (head_idx_within_batch_chunk) // ratio
        return (bch // _nh) * _ng + (bch % _nh) // _r

    Y_diag_rows = []

    for pm_chunk in range(PM):
        # ── How many K-tiles does this row-chunk touch? ───────────────────────
        # Tile k is ABOVE the diagonal when k*BK >= (pm_chunk+1)*BM (all K cols > all M rows).
        # K_active = ceil((pm_chunk+1)*BM / BK) — the first K_active tiles are on/below diagonal.
        # Integer ceil without math.ceil: (a + b - 1) // b
        K_active = min(((pm_chunk + 1) * _BM + _BK - 1) // _BK, PK)

        # ── Pre-slice: only K_active K-tiles, row pre-sliced to pm_chunk ──────
        # All of these are STATIC JAX slices (Python-int indices) → no dynamic
        # indexing inside the kernel, no lax.slice issues.

        # CB K-tiles for this row-chunk: shape (BCG, BM, BK)
        # Rows [pm_chunk*BM : (pm_chunk+1)*BM] × cols [k*BK : (k+1)*BK]
        CB_row_tiles  = [CB_flat[:, pm_chunk*_BM:(pm_chunk+1)*_BM, k*_BK:(k+1)*_BK]
                         for k in range(K_active)]

        # x K-tiles: shape (BCH, BK, hdim) — same for all pm_chunks (x has no M dependence)
        x_row_tiles   = [x_flat[:, k*_BK:(k+1)*_BK, :]    for k in range(K_active)]

        # dt, dA cumsum at K positions: shape (BCH, BK)
        dt_row_tiles  = [dt_flat[:, k*_BK:(k+1)*_BK]       for k in range(K_active)]
        dAk_row_tiles = [dA_flat[:, k*_BK:(k+1)*_BK]       for k in range(K_active)]

        # dA cumsum at M positions for this row-chunk: shape (BCH, BM)
        dA_m_row = dA_flat[:, pm_chunk*_BM:(pm_chunk+1)*_BM]

        # ── BlockSpecs for 2D grid (BCH, PN) ─────────────────────────────────
        # Each lambda takes (bch, pn). pm_chunk is fixed → not in the grid.
        in_specs_row = []
        for _ in range(K_active):
            # CB K-tile: (BCG, BM, BK) → block (1, BM, BK) at (bcg, 0, 0)
            # Row already pre-sliced → index 0 in dim 1 is the correct starting row.
            in_specs_row.append(pl.BlockSpec((1, _BM, _BK), lambda bch, pn: (_bcg(bch, pn), 0, 0)))
            # x K-tile: (BCH, BK, hdim) → block (1, BK, BN) at (bch, 0, pn)
            in_specs_row.append(pl.BlockSpec((1, _BK, _BN), lambda bch, pn: (bch, 0, pn)))
            # dt K-tile: (BCH, BK) → block (1, BK) at (bch, 0)
            in_specs_row.append(pl.BlockSpec((1, _BK),       lambda bch, pn: (bch, 0)))
            # dAk K-tile: (BCH, BK) → block (1, BK) at (bch, 0)
            in_specs_row.append(pl.BlockSpec((1, _BK),       lambda bch, pn: (bch, 0)))
        # dA_m: (BCH, BM) → block (1, BM) at (bch, 0)
        in_specs_row.append(pl.BlockSpec((1, _BM), lambda bch, pn: (bch, 0)))

        # Output: (BCH, BM, hdim) → block (1, BM, BN) at (bch, 0, pn)
        out_spec_row = [pl.BlockSpec((1, _BM, _BN), lambda bch, pn: (bch, 0, pn))]

        # ── Assemble argument list: [CB0,x0,dt0,dAk0, CB1,x1,dt1,dAk1, ..., dA_m] ──
        k_tile_args = []
        for k in range(K_active):
            k_tile_args.extend([
                CB_row_tiles[k],   # (BCG, BM, BK) — only K_active of these exist
                x_row_tiles[k],    # (BCH, BK, hdim)
                dt_row_tiles[k],   # (BCH, BK)
                dAk_row_tiles[k],  # (BCH, BK)
            ])
        k_tile_args.append(dA_m_row)  # (BCH, BM) — last input

        # ── Launch pallas_call for this row-chunk ─────────────────────────────
        # Grid: (BCH, PN) — fully parallel over batch-chunk-heads and N-tiles.
        # Output shape: (BCH, BM, hdim) — one BM-row-chunk of Y_diag.
        Y_row, = pl.pallas_call(
            _make_y_diag_kernel_v3(pm_chunk, K_active, _BM, _BK, _BN),
            out_shape=[jax.ShapeDtypeStruct((BCH, _BM, hdim), jnp.float32)],
            in_specs=in_specs_row,
            out_specs=out_spec_row,
            grid=(BCH, PN),
            compiler_params=CompilerParams(),
        )(*k_tile_args)

        Y_diag_rows.append(Y_row)   # accumulate PM row-chunks

    # ── Combine and reshape ───────────────────────────────────────────────────
    # Concatenate PM row-chunks along the M axis: (BCH, chunk_size, hdim)
    Y_diag = jnp.concatenate(Y_diag_rows, axis=1)

    # Add inter-chunk term and reshape to (batch, seqlen, nheads, hdim)
    Y = (Y_off + Y_diag).reshape(batch, nchunks, nheads, chunk_size, hdim)
    Y = Y.transpose(0, 1, 3, 2, 4).reshape(batch, seqlen, nheads, hdim)
    return Y.astype(jnp.bfloat16)


print("chunk_scan_pallas_v3 defined  "
      "(per-row-chunk dispatch, K_active K-tiles per call, static causal mask)")


chunk_scan_pallas_v3 defined  (per-row-chunk dispatch, K_active K-tiles per call, static causal mask)


In [20]:

# ── v3 correctness ────────────────────────────────────────────────────────────
# Note on v3 vs v2 difference: v2 uses a runtime-traced `m_abs = pm*BM + arange(BM)`
# in its causal mask (pm is a JAX tracer from pl.program_id). v3 has compile-time
# constants `m_abs = pm_chunk*BM + arange(BM)` (pm_chunk is a Python int), allowing
# XLA to fold the mask away entirely for fully-causal tiles. This produces numerically
# identical float32 results for normal values, but can produce bfloat16 differences
# due to different code generation paths. Primary correctness check is vs Triton (< 2.0).

print("=== Standard config (B=2, H=8, Q=64, ngroups=1) ===")
batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = 2, 512, 8, 64, 64, 1, 64
inp = make_inputs(batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size)
CB, x, dt, dA, C, B, states = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

Y_ref = jnp.array(
    _chunk_scan_fwd(to_torch_f32(CB), to_torch_bf16(x), to_torch_f32(dt),
                    to_torch_f32(dA), to_torch_bf16(C), to_torch_f32(states))[0]
    .float().cpu().numpy()
).astype(jnp.bfloat16)

Y_naive = jax.jit(lambda *a: chunk_scan_naive(*a))(CB, x, dt, dA, C, states)
Y_v2    = jax.jit(lambda *a: chunk_scan_pallas_v2(*a, BLOCK_M=32, BLOCK_K=32, BLOCK_N=64))(CB, x, dt, dA, C, states)
Y_v3    = jax.jit(lambda *a: chunk_scan_pallas_v3(*a, BLOCK_M=32, BLOCK_K=32, BLOCK_N=64))(CB, x, dt, dA, C, states)

d_v3_tri   = float(jnp.max(jnp.abs(Y_v3.astype(jnp.float32) - Y_ref.astype(jnp.float32))))
d_v3_naive = float(jnp.max(jnp.abs(Y_v3.astype(jnp.float32) - Y_naive.astype(jnp.float32))))
d_v3_v2    = float(jnp.max(jnp.abs(Y_v3.astype(jnp.float32) - Y_v2.astype(jnp.float32))))
print(f"  v3 vs Triton: {d_v3_tri:.4f}   (primary check, threshold 2.0)")
print(f"  v3 vs naive:  {d_v3_naive:.4f}   (secondary check, threshold 2.0)")
print(f"  v3 vs v2:     {d_v3_v2:.4f}   (informational — may differ due to compile-time mask folding)")
assert d_v3_tri   < 2.0, f"v3 vs Triton: {d_v3_tri}"
assert d_v3_naive < 2.0, f"v3 vs naive: {d_v3_naive}"
print("  OK")

print("\n=== Nemotron config (B=1, H=128, Q=256, ngroups=8) ===")
inp2 = make_inputs(1, 256, 128, 64, 128, 8, 256)
CB2, x2, dt2, dA2, C2, B2, st2 = [inp2[k] for k in ('CB','x','dt','dA','C','B','states')]

Y_ref2 = jnp.array(
    _chunk_scan_fwd(to_torch_f32(CB2), to_torch_bf16(x2), to_torch_f32(dt2),
                    to_torch_f32(dA2), to_torch_bf16(C2), to_torch_f32(st2))[0]
    .float().cpu().numpy()
).astype(jnp.bfloat16)

Y_naive2 = jax.jit(lambda *a: chunk_scan_naive(*a))(CB2, x2, dt2, dA2, C2, st2)
Y_v2_nem = jax.jit(lambda *a: chunk_scan_pallas_v2(*a, BLOCK_M=64, BLOCK_K=64, BLOCK_N=64))(CB2, x2, dt2, dA2, C2, st2)
Y_v3_nem = jax.jit(lambda *a: chunk_scan_pallas_v3(*a, BLOCK_M=64, BLOCK_K=64, BLOCK_N=64))(CB2, x2, dt2, dA2, C2, st2)

d_v3_tri2   = float(jnp.max(jnp.abs(Y_v3_nem.astype(jnp.float32) - Y_ref2.astype(jnp.float32))))
d_v3_naive2 = float(jnp.max(jnp.abs(Y_v3_nem.astype(jnp.float32) - Y_naive2.astype(jnp.float32))))
d_v3_v2_2   = float(jnp.max(jnp.abs(Y_v3_nem.astype(jnp.float32) - Y_v2_nem.astype(jnp.float32))))
print(f"  v3 vs Triton: {d_v3_tri2:.4f}   (primary check, threshold 2.0)")
print(f"  v3 vs naive:  {d_v3_naive2:.4f}   (secondary check, threshold 2.0)")
print(f"  v3 vs v2:     {d_v3_v2_2:.4f}   (informational)")
assert d_v3_tri2   < 2.0, f"v3 vs Triton Nemotron: {d_v3_tri2}"
assert d_v3_naive2 < 2.0, f"v3 vs naive Nemotron: {d_v3_naive2}"
print("  OK")

# ── v3 benchmark ──────────────────────────────────────────────────────────────

def _best_block_v3(chunk_size, hdim):
    """
    v3 optimal block sizes.
    BM = BK: square K-tiles give maximum triangular bandwidth savings AND keep
    the causal mask square (simplest masking logic). For BM > BK, K_active grows
    faster per row-chunk → savings diminish. For BM < BK, savings plateau at
    the same ~37.5% (for PM=PK=4) but launch overhead increases.
    BN: as large as fits in hdim.
    """
    tile = max(b for b in [32, 64] if chunk_size % b == 0)
    bn   = max(b for b in [32, 64] if hdim    % b == 0)
    return tile, tile, bn  # BM=BK=tile, BN

W = 100
print("\n" + "═" * W)
print("SCAN-ONLY benchmark: Triton vs Naive vs v1 vs v2 vs v3  (CB pre-computed, not counted)")
print(f"{'Config':<45} {'Triton':>7} {'Naive':>7} {'v1':>7} {'v2':>7} {'v3':>7} {'v3/Tri':>8}")
print(f"{'':45} {'(ms)':>7} {'(ms)':>7} {'(ms)':>7} {'(ms)':>7} {'(ms)':>7} {'speedup':>8}")
print("─" * W)

for cfg, label in SHAPE_SWEEP:
    batch, seqlen, nheads, hdim, dstate, ngroups, chunk_size = cfg
    inp = make_inputs(*cfg)
    CB, x, dt, dA, C, B, st = [inp[k] for k in ('CB','x','dt','dA','C','B','states')]

    # Triton
    CB_t = to_torch_f32(CB); x_t = to_torch_bf16(x)
    dt_t = to_torch_f32(dt); dA_t = to_torch_f32(dA)
    C_t  = to_torch_bf16(C); st_t = to_torch_f32(st)
    t_tri = bench_triton_amortized(
        lambda: _chunk_scan_fwd(CB_t, x_t, dt_t, dA_t, C_t, st_t), (), N_AMORTIZE
    )

    # Naive JAX
    fn_naive = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj: chunk_scan_naive(cb,xj,dtj,daj,cj,sj), N_AMORTIZE
    )
    t_naive = bench_amortized(fn_naive, (CB,x,dt,dA,C,st), N_AMORTIZE)

    # v1 Pallas
    BM1, BN1 = _best_block(chunk_size, hdim)
    fn_v1 = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj,BM=BM1,BN=BN1:
            chunk_scan_pallas(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_N=BN),
        N_AMORTIZE
    )
    t_v1 = bench_amortized(fn_v1, (CB,x,dt,dA,C,st), N_AMORTIZE)

    # v2 Pallas
    BM2, BK2, BN2 = _best_block_v2(chunk_size, hdim)
    fn_v2 = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj,BM=BM2,BK=BK2,BN=BN2:
            chunk_scan_pallas_v2(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_K=BK,BLOCK_N=BN),
        N_AMORTIZE
    )
    t_v2 = bench_amortized(fn_v2, (CB,x,dt,dA,C,st), N_AMORTIZE)

    # v3 Pallas  — per-row-chunk, only K_active tiles loaded per call
    BM3, BK3, BN3 = _best_block_v3(chunk_size, hdim)
    fn_v3 = make_amortized(
        lambda cb,xj,dtj,daj,cj,sj,BM=BM3,BK=BK3,BN=BN3:
            chunk_scan_pallas_v3(cb,xj,dtj,daj,cj,sj,BLOCK_M=BM,BLOCK_K=BK,BLOCK_N=BN),
        N_AMORTIZE
    )
    t_v3 = bench_amortized(fn_v3, (CB,x,dt,dA,C,st), N_AMORTIZE)

    speedup = f"{t_tri/t_v3:.2f}x"
    print(f"{label:<45} {t_tri:>7.3f} {t_naive:>7.3f} {t_v1:>7.3f} {t_v2:>7.3f} {t_v3:>7.3f} {speedup:>8}")

print("═" * W)
print("(v3/Tri speedup > 1.0x means v3 faster than Triton)")


=== Standard config (B=2, H=8, Q=64, ngroups=1) ===


  v3 vs Triton: 0.5000   (primary check, threshold 2.0)
  v3 vs naive:  0.1250   (secondary check, threshold 2.0)
  v3 vs v2:     0.1250   (informational — may differ due to compile-time mask folding)
  OK

=== Nemotron config (B=1, H=128, Q=256, ngroups=8) ===


  v3 vs Triton: 1.0000   (primary check, threshold 2.0)
  v3 vs naive:  0.1250   (secondary check, threshold 2.0)
  v3 vs v2:     0.0625   (informational)
  OK

════════════════════════════════════════════════════════════════════════════════════════════════════
SCAN-ONLY benchmark: Triton vs Naive vs v1 vs v2 vs v3  (CB pre-computed, not counted)
Config                                         Triton   Naive      v1      v2      v3   v3/Tri
                                                 (ms)    (ms)    (ms)    (ms)    (ms)  speedup
────────────────────────────────────────────────────────────────────────────────────────────────────


Mamba2 standard   B=1 H=32  P=64  N=64  Q=64    0.102   0.087   0.051   0.047   0.047    2.15x


Mamba2 cs=256     B=1 H=32  P=64  N=64  Q=256   0.102   0.187   0.132   0.072   0.085    1.20x


Nemotron          B=1 H=128 P=64  N=128 Q=256   0.188   1.330   0.863   0.796   0.787    0.24x


Batched           B=2 H=32  P=64  N=64  Q=64    0.085   0.312   0.129   0.210   0.210    0.41x
════════════════════════════════════════════════════════════════════════════════════════════════════
(v3/Tri speedup > 1.0x means v3 faster than Triton)
